# LLaVA-Med 7B 베이스라인 재현 — VQA-RAD + SLAKE (delta 병합 방식)

논문: *LLaVA-Med: Training a Large Language-and-Vision Assistant for Biomedicine in One Day*
(arXiv:2306.00890) — 생성형(autoregressive) MedVQA.

**downstream fine-tuning 을 직접 하지 않고**, 데이터셋별로 이미 fine-tuning 된 공개 delta 를 LLaMA-7B 에 병합해서 평가합니다.

**재현 목표 (Expert-CFG 논문 Table 1 기준)**

| 데이터셋 | Open | Closed | Overall |
|---|---|---|---|
| VQA-RAD | 61.52 | 84.19 | 75.19 |
| SLAKE   | 83.08 | 85.34 | 83.97 |

Repo: https://github.com/microsoft/LLaVA-Med (**v1.0.0 태그**) ·
Delta: [vqarad](https://huggingface.co/katielink/llava-med-7b-vqarad-delta) · [slake](https://huggingface.co/katielink/llava-med-7b-slake-delta) (microsoft-research-license — **연구 목적 한정**)

> **SLAKE delta 는 현재 받을 수 없습니다.** `katielink/llava-med-7b-slake-delta` 는 저장소만 있고 가중치 파일이 없으며(README 만 존재, 2023-11 최초 커밋 이후 변경 없음), LLaVA-Med README 가 가리키는 Azure 링크(`Slake1.0-9epoch_delta.zip`)도 `WebsiteDisabled` 로 막혀 있습니다. VQA-RAD(`katielink/llava-med-7b-vqarad-delta`)는 가중치가 있습니다. SLAKE 는 아래 절에서 건너뛰도록 되어 있고, 대안은 팀과 상의가 필요합니다.

## 구성
| 절 | 내용 |
|---|---|
| 0. 환경 준비 | repo(v1.0.0) + **Python 3.10 venv** (torch 2.0.0, transformers 고정 커밋) |
| 1. LLaMA-7B 베이스 | `huggyllama/llama-7b` 또는 Drive 사본 |
| 2. delta 다운로드 | HF 에서 두 delta |
| 3. delta 병합 | `llava.model.apply_delta` |
| 4. 평가 데이터 준비 | `model_vqa_med.py` 입력 형식(JSON 배열 + `conversations`) + 채점용 참조 파일 |
| 5~6. 추론 | VQA-RAD, SLAKE |
| 7. 채점 | Phi-3.5 노트북과 같은 open recall / closed accuracy |
| 8. 결과 요약 | 논문 수치와 비교 |

**코드를 읽고 확인한 사항 (실행해 본 것은 아님)**
- `model_vqa_med.py` 의 질문 파일은 **JSON 배열**이고 각 항목에 `id`, `conversations`(`[0]` human 질문, `[1]` from=gpt 정답), 선택적으로 `image`(= `--image-folder` 기준 상대 경로)가 있어야 합니다. 출력 jsonl 키는 `question_id`(= 입력 `id`), `text`, `prompt` …
- `pyproject.toml` 이 `transformers @ git+…@cae78c46`, `tokenizers==0.12.1`, `gradio==3.23` 을 요구해서 Colab 기본 환경(Python 3.12)에는 설치되지 않습니다 → 별도 venv.
- `apply_delta` 는 base 와 delta 를 fp16 으로 메모리에 올립니다. **CPU RAM 이 넉넉해야** 합니다 (고용량 RAM 런타임 권장). 디스크: LLaMA 13GB + delta 2개 + 병합본 2개 ≈ 70GB.
- SLAKE 이미지는 HF `BoKelvin/SLAKE` 의 `imgs.zip`(`imgs/xmlabN/source.jpg`)을 쓰고, `image` 는 `xmlabN/source.jpg`, `--image-folder` 는 `imgs/` 입니다. 영어 test 는 1061 (OPEN 645 / CLOSED 416) 이어야 합니다.

## 0. 환경 준비

In [18]:
import subprocess, os
subprocess.run('rm -rf /content/LLaVA-Med', shell=True)
subprocess.run(['git', 'clone', 'https://github.com/microsoft/LLaVA-Med.git', '/content/LLaVA-Med'], check=True)
r = subprocess.run(['git', 'checkout', 'v1.0.0'], cwd='/content/LLaVA-Med', capture_output=True, text=True)
print('returncode:', r.returncode); print(r.stderr)
print(subprocess.run(['git', 'describe', '--tags'], cwd='/content/LLaVA-Med', capture_output=True, text=True).stdout)


returncode: 0
Note: switching to 'v1.0.0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at b9a98a7 fix closed accuracy calculation (#35)

v1.0.0



In [ ]:
# Python 3.10 venv (v1.0.0 의 의존성은 Colab 기본 Python 에서 설치되지 않음). 실패하면 여기서 멈춘다.
# torch 는 2.0.0 으로 고정한다 (고정하지 않으면 -e . 설치 중 최신 torch(cu130)로 올라가 bitsandbytes 0.41 이 CUDA 설정에 실패한다)
!pip install -q uv
!uv venv /content/venv_llava --python 3.10
open('/content/constraints_llava.txt', 'w').write('torch==2.0.0' + chr(10) + 'torchvision==0.15.1' + chr(10))      # torch 고정 파일
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt torch==2.0.0 torchvision==0.15.1
!cd /content/LLaVA-Med && uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt -e .
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt einops shortuuid ninja open-clip-torch==2.24.0 "numpy<2"
!/content/venv_llava/bin/python -c "import torch, transformers, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"


SyntaxError: unterminated string literal (detected at line 7) (1131144403.py, line 7)

In [14]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import os
DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'
os.makedirs(DRIVE_ROOT, exist_ok=True)
PY = '/content/venv_llava/bin/python'
print('OK')

Mounted at /content/drive
OK


In [15]:
!uv pip install --python /content/venv_llava/bin/python "numpy<2"
!/content/venv_llava/bin/python -c "import numpy, torch; print(numpy.__version__, torch.__version__); print(torch.from_numpy(numpy.zeros(2)))"


Using Python 3.10.12 environment at: venv_llava
Checked 1 package in 2ms
1.26.4 2.14.1+cu130
tensor([0., 0.], dtype=torch.float64)


## 1. LLaMA-7B 베이스 확보

delta 는 차이값이라 원본 LLaMA-7B(HF 포맷) 위에 더해야 합니다. Drive 에 변환본이 있으면 `LLAMA_BASE` 로 쓰고,
없으면 HF 공개 변환본 `huggyllama/llama-7b` 를 받습니다 (라이선스 확인 후 사용).

In [16]:
import os
from huggingface_hub import snapshot_download

LLAMA_BASE = '/content/drive/MyDrive/llama-7b-hf'
if not os.path.isdir(LLAMA_BASE):
    print(f'{LLAMA_BASE} 없음 → huggyllama/llama-7b 다운로드')
    LLAMA_BASE = snapshot_download('huggyllama/llama-7b', local_dir='/content/llama-7b-hf',
                                   allow_patterns=['*.json', '*.model', '*.safetensors'])
print('LLAMA_BASE =', LLAMA_BASE)
assert os.path.exists(os.path.join(LLAMA_BASE, 'config.json')), 'LLaMA-7B 가중치가 올바르지 않습니다'
assert any(f.endswith(('.safetensors', '.bin')) for f in os.listdir(LLAMA_BASE)), '가중치 파일이 없습니다 (allow_patterns 확인)'

/content/drive/MyDrive/llama-7b-hf 없음 → huggyllama/llama-7b 다운로드


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LLAMA_BASE = /content/llama-7b-hf


## 2. delta 다운로드

In [19]:
from huggingface_hub import snapshot_download

VQARAD_DELTA = snapshot_download('katielink/llava-med-7b-vqarad-delta', local_dir='/content/delta_vqarad')
SLAKE_DELTA  = snapshot_download('katielink/llava-med-7b-slake-delta',  local_dir='/content/delta_slake')   # 가중치가 없는 빈 저장소(README 만)
print('VQARAD_DELTA =', VQARAD_DELTA)
print('SLAKE_DELTA  =', SLAKE_DELTA)
SLAKE_AVAILABLE = os.path.exists(os.path.join(SLAKE_DELTA, 'config.json'))
print('SLAKE delta 사용 가능:', SLAKE_AVAILABLE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

VQARAD_DELTA = /content/delta_vqarad
SLAKE_DELTA  = /content/delta_slake
SLAKE delta 사용 가능: False


In [25]:
!uv pip install --python /content/venv_llava/bin/python -c /content/constraints_llava.txt "transformers @ git+https://github.com/huggingface/transformers@cae78c46" tokenizers==0.12.1 "numpy<2"
!cd /content/LLaVA-Med && /content/venv_llava/bin/python -c "import torch, transformers, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"


Using Python 3.10.12 environment at: /content/venv_llava
Resolved 17 packages in 14.51s                                       
Prepared 2 packages in 2.28s                                             
Uninstalled 2 packages in 45ms
Installed 2 packages in 24msdev0 (from git+https://github.co
 - tokenizers==0.15.2
 + tokenizers==0.12.1
 - transformers==4.36.2
 + transformers==4.28.0.dev0 (from git+https://github.com/huggingface/transformers@cae78c46d658a8e496a815c2ee49b9b178fb9c9a)
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2.0.0+cu117 4.28.0.dev0 True


## 3. delta 병합

venv 의 파이썬으로 실행합니다(인자 이름은 `--base-model-path / --target-model-path / --delta-path`).
병합이 끝나면 delta 는 지워도 됩니다(디스크 절약).

In [26]:
%cd /content/LLaVA-Med
!{PY} -m llava.model.apply_delta \
    --base-model-path {LLAMA_BASE} \
    --target-model-path /content/llava-med-7b-vqarad \
    --delta-path {VQARAD_DELTA}
assert os.path.exists('/content/llava-med-7b-vqarad/config.json'), '병합 실패'

/content/LLaVA-Med
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading base model

Loading delta
/content/venv_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModel: ['text_model.encoder.layers.6.mlp.fc1.bias', 'text_model.encoder.layers.2.mlp.fc1.weight', 'text_model.encoder.layers.10.self_attn.out_proj.weight', 'text_model.encoder.layers.2.self_attn.out_proj.weight', 'text_model.encoder.layers.2.self_attn.q_pro

## 4. 평가 데이터 준비

`model_vqa_med.py` 입력: **JSON 배열**, 항목 = `{id, image, conversations:[{from:human,value:질문},{from:gpt,value:정답}]}`.
채점용 참조(`answer`, `answer_type`)는 `*_ref.json` 으로 따로 저장합니다.
VQA-RAD 는 Phi-3.5 노트북과 같은 451문항입니다.
**closed 문항(`answer_type == CLOSED`)에는 질문 뒤에 ` Please choose from the following two options: [yes, no]` 를 붙입니다** (레포 샘플 프롬프트 형식). open 문항은 질문만 보냅니다.


In [28]:
import os, json, shutil, subprocess

EVAL_DIR = '/content/llava_eval_data'
os.makedirs(EVAL_DIR, exist_ok=True)

_GDRIVE_JSON   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

# LLaVA-Med 레포의 closed(yes/no) 문항 프롬프트 형식 (llava/eval/eval_metrics/answer-file.jsonl 의 prompt 필드).
# 파인튜닝 데이터가 이 형식이었다면 접미어 없이 물으면 yes/no 로 답하지 않아 closed 점수가 크게 떨어진다.
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'

def write_eval_files(name, rows, get_image):
    """rows: dict(question, answer, answer_type, ...). 질문 JSON 배열 + 참조 JSON 저장."""
    questions, refs = [], {}
    for i, r in enumerate(rows):
        questions.append({
            'id': i,
            'image': get_image(r),
            'conversations': [{'from': 'human', 'value': r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')},
                              {'from': 'gpt',   'value': str(r['answer'])}],
        })
        refs[i] = {'answer': str(r['answer']), 'answer_type': r['answer_type']}
    json.dump(questions, open(f'{EVAL_DIR}/{name}_test.json', 'w'))
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')

# ── VQA-RAD ──
_raw_json = _GDRIVE_JSON
if not os.path.exists(_raw_json):
    _raw_json = os.path.join(EVAL_DIR, 'VQA_RAD_Dataset_Public.json')
    subprocess.run(['wget', '-q', '-O', _raw_json, 'https://osf.io/download/9dpze/'], timeout=120, check=True)
all_data = json.load(open(_raw_json))
for item in all_data:
    item['answer_type'] = item.get('answer_type', answer_type_of(item.get('answer', ''))).upper()
_TEST_TYPES = {'test_freeform', 'test_para'}
assert _TEST_TYPES & {str(r.get('phrase_type', '')).lower() for r in all_data}, 'phrase_type 에 test 표시가 없습니다'
vqarad_test = [r for r in all_data if str(r.get('phrase_type', '')).lower() in _TEST_TYPES]
assert len(vqarad_test) == 451, len(vqarad_test)

VQARAD_IMG_DIR = _GDRIVE_IMGDIR
assert os.path.isdir(VQARAD_IMG_DIR), f'이미지 폴더 없음: {VQARAD_IMG_DIR}'
write_eval_files('vqarad', vqarad_test, lambda r: r['image_name'])
assert all(os.path.exists(os.path.join(VQARAD_IMG_DIR, r['image_name'])) for r in vqarad_test), '이미지 누락'

vqarad: 451문항 (OPEN 179 / CLOSED 272)


In [12]:
# ── SLAKE (영어 test) — HF 원본 test.json + imgs.zip ──
import zipfile
from huggingface_hub import hf_hub_download

_p = hf_hub_download('BoKelvin/SLAKE', 'test.json', repo_type='dataset')
slake_test = [r for r in json.load(open(_p)) if r['q_lang'] == 'en']
assert len(slake_test) == 1061, len(slake_test)

SLAKE_ROOT = '/content/slake_data'
if not os.path.isdir(f'{SLAKE_ROOT}/imgs'):
    _zip = hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')
    with zipfile.ZipFile(_zip) as z:
        z.extractall(SLAKE_ROOT)
SLAKE_IMG_DIR = f'{SLAKE_ROOT}/imgs'
for r in slake_test:
    r['answer_type'] = r['answer_type'].upper()
write_eval_files('slake', slake_test, lambda r: r['img_name'])
assert all(os.path.exists(os.path.join(SLAKE_IMG_DIR, r['img_name'])) for r in slake_test), '이미지 누락'

test.json: 0.00B [00:00, ?B/s]

imgs.zip:   0%|          | 0.00/212M [00:00<?, ?B/s]

slake: 1061문항 (OPEN 645 / CLOSED 416)


In [29]:
import json
_qs = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))[:20]
json.dump(_qs, open(f'{EVAL_DIR}/vqarad_smoke.json', 'w'))
%cd /content/LLaVA-Med
!{PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_smoke.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file /content/vqarad_smoke_pred.jsonl
!head -n 5 /content/vqarad_smoke_pred.jsonl | cut -c1-300


/content/LLaVA-Med
/content/venv_llava/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/llava-med-7b-vqarad
/content/venv_llava/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPVisionModel: ['text_model.encoder.layers.9.layer_norm2.weight', 'text_model.encoder.layers.4.self_attn.out_proj.weight', 'text_model.encoder.layers.10.mlp.fc1.bias', 'text_model.encoder.layers.6.self_attn.q_proj.bias', 'text_model.encoder.layers.5.layer_norm1.weight', '

## 5. 추론 — VQA-RAD

한 문항씩 생성합니다. 질문 파일 앞부분만 잘라 먼저 스모크 테스트를 해보길 권합니다 (오래 걸리면 `--num-chunks`/`--chunk-idx` 로 나눔).

In [15]:
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_test.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred.jsonl \
    > /content/vqarad_infer.log 2>&1 &


/content


## 6. 추론 — SLAKE

In [ ]:
%cd /content/LLaVA-Med
!{PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-slake \
    --question-file {EVAL_DIR}/slake_test.json \
    --image-folder {SLAKE_IMG_DIR} \
    --answers-file {DRIVE_ROOT}/slake_pred.jsonl

In [85]:
# 1) closed 문항만 분리하면서 접미어를 직접 붙임
import json
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
closed_qs = []
for q in qs:
    if ref[str(q['id'])]['answer_type'] == 'CLOSED':
        v = q['conversations'][0]['value']
        if not v.endswith(CLOSED_SUFFIX):
            q['conversations'][0]['value'] = v + CLOSED_SUFFIX
        closed_qs.append(q)
json.dump(closed_qs, open(f'{EVAL_DIR}/vqarad_closed.json', 'w'))
print(len(closed_qs), '|', closed_qs[0]['conversations'][0]['value'])

# 2) 백그라운드 추론 (기존 open 결과는 건드리지 않음)
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_closed.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred_closed2.jsonl \
    > /content/vqarad_closed2.log 2>&1 &


272 | Is there evidence of an aortic aneurysm? Please choose from the following two options: [yes, no]
/content


In [ ]:
import time, os
_pred = f'{DRIVE_ROOT}/vqarad_pred_closed2.jsonl'
n = sum(1 for _ in open(_pred)) if os.path.exists(_pred) else 0
print(f'{n}/451 완료')
!tail -c 400 /content/vqarad_infer.log | tr '\r' '\n' | tail -n 3
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader


272/451 완료
100%|██████████| 451/451 [03:40<00:00,  2.04it/s]
0 %, 14876 MiB


In [74]:
!pgrep -f model_vqa_med.py || echo "추론 프로세스 종료"


37528
38099


In [ ]:
# 3) 끝난 뒤: 기존 open 예측 + 새 closed 예측을 합쳐 채점
import json
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred.jsonl')]
new = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed2.jsonl')]
assert len(new) == 272, len(new)
new_ids = {p['question_id'] for p in new}
merged = [p for p in old if p['question_id'] not in new_ids] + new
with open(f'{DRIVE_ROOT}/vqarad_pred_v2.jsonl', 'w') as f:
    for p in merged: f.write(json.dumps(p) + '\n')
vqarad_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_v2.jsonl')
print(vqarad_m)


{'open': 64.50784783187018, 'closed': 70.22058823529412, 'overall': 67.95322563615245, 'n_open': 179, 'n_closed': 272}


## 7. 채점

레포의 `run_eval.py` 는 논문 Table 1 의 recall/accuracy 와 정의가 다르므로, Phi-3.5 노트북과 **같은 open recall / closed accuracy 함수**로 직접 채점합니다.

In [104]:
import json, re
import numpy as np
from collections import Counter

def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def score_jsonl(ref_file, pred_file):
    refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
    preds = {}
    for line in open(pred_file):
        p = json.loads(line)
        preds[int(p['question_id'])] = p['text']
    assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개 — 추론이 끝나지 않았거나 id 불일치'
    cl, op = [], []
    for i, ref in refs.items():
        if ref['answer_type'] == 'CLOSED':
            cl.append(closed_accuracy(preds[i], ref['answer']))
        else:
            op.append(open_recall(preds[i], ref['answer']))
    return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op),
            'n_open': len(op), 'n_closed': len(cl)}

vqarad_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred.jsonl')
_sp = f'{DRIVE_ROOT}/slake_pred.jsonl'
slake_m  = score_jsonl(f'{EVAL_DIR}/slake_ref.json', _sp) if os.path.exists(_sp) else None   # SLAKE delta 없으면 None
print('VQA-RAD:', vqarad_m)
print('SLAKE  :', slake_m)

VQA-RAD: {'open': 64.50784783187018, 'closed': 20.955882352941178, 'overall': 38.241473973181286, 'n_open': 179, 'n_closed': 272}
SLAKE  : None


## 8. 결과 요약

In [108]:
paper = {
    'VQA-RAD': {'open': 61.52, 'closed': 84.19, 'overall': 75.19},
    'SLAKE':   {'open': 83.08, 'closed': 85.34, 'overall': 83.97},
}
ours = {'VQA-RAD': vqarad_m, 'SLAKE': slake_m}

print(f"{'':10s}{'open':>10s}{'closed':>10s}{'overall':>10s}")
for ds in ('VQA-RAD', 'SLAKE'):
    p, o = paper[ds], ours[ds]
    if o is None:
        print(f'[{ds}] 결과 없음'); continue
    print(f'[{ds}]')
    print(f"  paper   {p['open']:10.2f}{p['closed']:10.2f}{p['overall']:10.2f}")
    print(f"  ours    {o['open']:10.2f}{o['closed']:10.2f}{o['overall']:10.2f}")
    print(f"  diff    {o['open']-p['open']:+10.2f}{o['closed']-p['closed']:+10.2f}{o['overall']-p['overall']:+10.2f}")

                open    closed   overall
[VQA-RAD]
  paper        61.52     84.19     75.19
  ours         64.51     70.22     67.95
  diff         +2.99    -13.97     -7.24
[SLAKE] 결과 없음


In [109]:
# 1) 레포 코드를 greedy 로 패치 (원본 문자열이 없으면 assert 로 알림)
import re
p = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
s = open(p).read()
pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(pat, s):
    s = re.sub(pat, "do_sample=False,\n                max_new_tokens=1024,", s, count=1)
    open(p, 'w').write(s)
    print('greedy 패치 완료')
else:
    assert 'do_sample=False,' in s, '패치 대상 문자열을 찾지 못했습니다'
    print('이미 패치됨')
!grep -n "do_sample" {p}



greedy 패치 완료
235:                do_sample=False,
278:                    do_sample=True,


In [110]:
# 2) 전체 451문항 질문 파일 (closed 에는 접미어) 생성
import json
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
for q in qs:
    v = q['conversations'][0]['value']
    if ref[str(q['id'])]['answer_type'] == 'CLOSED' and not v.endswith(CLOSED_SUFFIX):
        q['conversations'][0]['value'] = v + CLOSED_SUFFIX
json.dump(qs, open(f'{EVAL_DIR}/vqarad_test_suffix.json', 'w'))
print(len(qs), '|', sum(q['conversations'][0]['value'].endswith(CLOSED_SUFFIX) for q in qs), '개 접미어 적용')   # 451 | 272


451 | 272 개 접미어 적용


In [111]:
# 3) greedy 추론 (백그라운드)
%cd /content/LLaVA-Med
!PYTHONUNBUFFERED=1 nohup {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_test_suffix.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {DRIVE_ROOT}/vqarad_pred_greedy.jsonl \
    > /content/vqarad_greedy.log 2>&1 &


/content


In [126]:
import os
_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred)) if os.path.exists(_pred) else 0
print(f'{n}/451 완료')
!tail -c 400 /content/vqarad_greedy.log | tr '\r' '\n' | tail -n 3
!pgrep -f model_vqa_med.py > /dev/null && echo "추론 실행 중" || echo "추론 프로세스 없음"


451/451 완료
100%|██████████| 451/451 [03:06<00:00,  2.42it/s]
추론 실행 중


In [127]:
import json, os
from collections import Counter
import numpy as np

_pred = f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
n = sum(1 for _ in open(_pred))
assert n == 451, f'예측 {n}/451 — 추론이 아직 끝나지 않았습니다'

# 프롬프트에 접미어가 실제로 들어갔는지 확인 (closed 는 접미어가 보여야 함)
first = [json.loads(l) for l in open(_pred)][:3]
print('prompt 확인:', [p['prompt'][-70:] for p in first])

greedy_m = score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', _pred)
print(greedy_m)


prompt 확인: ['urysm? Please choose from the following two options: [yes, no]\n<image>', ' side? Please choose from the following two options: [yes, no]\n<image>', 'plane? Please choose from the following two options: [yes, no]\n<image>']
{'open': 66.08140462889067, 'closed': 72.05882352941177, 'overall': 69.68641114982577, 'n_open': 179, 'n_closed': 272}


In [182]:
# 2) 백그라운드 추론 (venv_llava)
import subprocess
_cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py '
        f'--model-name /content/llava-med-7b-vqarad --question-file {EVAL_DIR}/vqarad_closed_alt.json '
        f'--image-folder "{VQARAD_IMG_DIR}" --answers-file {DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl')
_log = open('/content/vqarad_closed_alt.log', 'w')
alt_proc = subprocess.Popen(['bash', '-c', _cmd], stdout=_log, stderr=subprocess.STDOUT, start_new_session=True)
print('PID', alt_proc.pid)




PID 56949


In [195]:
import os
_p = f'{DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl'
n = sum(1 for _ in open(_p)) if os.path.exists(_p) else 0
print(f'{n}/272 완료 | 프로세스 종료 코드(None=실행 중):', alt_proc.poll())
!tail -n 3 /content/vqarad_closed_alt.log | tr '\r' '\n' | tail -n 2 | cut -c1-200


245/272 완료 | 프로세스 종료 코드(None=실행 중): 1
 90%|█████████ | 245/272 [01:33<00:09,  2.79it/s]


In [202]:
# 3) 끝난 뒤: 기존 greedy open 예측 + 새 closed 예측을 합쳐 채점
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')]
new = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed_altfmt.jsonl')]
assert len(new) == 272, f'closed 예측 {len(new)}개 — 추론이 아직 끝나지 않았습니다'
new_ids = {p['question_id'] for p in new}
merged = [p for p in old if p['question_id'] not in new_ids] + new
with open(f'{DRIVE_ROOT}/vqarad_pred_altfmt.jsonl', 'w') as f:
    for p in merged: f.write(json.dumps(p) + '\n')
print(score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_altfmt.jsonl'))


{'open': 66.08140462889067, 'closed': 75.0, 'overall': 71.46024707000316, 'n_open': 179, 'n_closed': 272}


---
# 부록 C — SLAKE QLoRA 파인튜닝 (SLAKE 공식 delta 를 받을 수 없을 때의 **근사 행**)

**SLAKE 에만 적용합니다.** VQA-RAD 는 공식 delta 로 평가했으므로 그대로 둡니다. 이 절의 결과는 논문의 LLaVA-Med(전체 파인튜닝)와 **같은 조건이 아닙니다** — 결과표에는 "QLoRA 근사"로 별도 표시하세요.

| 항목 | 설정 |
|---|---|
| 출발점 | `microsoft/llava-med-7b-delta` + LLaMA-7B 병합 = LLaVA-Med 의료 instruct 모델 (v1.0, 다운스트림 파인튜닝 전) |
| 방식 | 4비트(NF4, double quant) + LoRA(r=64, alpha=16, dropout 0.05, LLaMA 의 선형층 7종). 비전 타워·`mm_projector`·`lm_head` 는 고정 |
| 학습률 | **2e-5 / 1e-4 / 2e-4 세 가지만 바꾸고** 나머지(유효 배치 16, 코사인, warmup 3%, 시드 42)는 동일 (학습 길이는 1단계에서 모든 학습률이 같다) |
| 데이터 | SLAKE 영어 **train 으로만 최적화**, **val 은 선택 전용**(학습률·epoch), **test 는 마지막 단계 전까지 읽지 않음** |
| 프롬프트 | 기존 평가 경로(`model_vqa_med.py`, `conv_mode=simple`)와 같은 형식. closed 질문에는 평가 때와 같은 접미어 |

**진행 순서**
0. **C-0a/b: 단독 실행용 준비**(Drive·레포·추론 venv·LLaMA 베이스·greedy 패치·이미지·채점 함수) — 앞 절 없이 여기서부터 실행해도 됩니다
1. 환경(별도 venv, transformers 4.31) · instruct 모델 병합 · 데이터 생성
2. **샌티 체크**: 학습률마다 40스텝 — 발산/불안정 확인용입니다. 이 손실로 학습률을 고르지 않습니다.
3. **1단계(학습률 탐색)**: 학습률마다 `EPOCHS`(기본 2) epoch. **2단계(선택, C-9c/9d)**: val 로 고른 학습률 하나만 `FINAL_EPOCHS`(기본 6; 공식 SLAKE delta 는 9 epoch 였으나 비용 때문에 줄임)로 길게. 3학습률 × 9 epoch 를 전부 돌리는 것보다 비용이 크게 준다. gradient checkpointing 은 기본으로 껐다(C-4 의 `GRAD_CKPT`; 메모리 부족이면 True). 매 epoch 끝에 val 손실과 **val 생성 결과**(closed accuracy, open recall/EM)를 기록합니다.
4. **선택**: train 손실이 아니라 **val 성능**(overall; 동점이면 val 손실)으로 학습률·epoch 선택
5. **test 는 선택된 설정에 대해서만(C-9b 에서 처음 읽음)** (val 선택 epoch 와 마지막 epoch 두 개) 추론·채점

> **아직 한 번도 실행해 보지 않은 코드입니다.** v1.0 코드는 오래된 transformers 에 묶여 있어서 4비트 로딩·LoRA 연결에서 막힐 수 있습니다. 2단계(샌티 체크)에서 오류가 나면 그 로그를 먼저 확인하세요.
> 시간: 비용은 샌티 체크(C-6)가 처리 속도로 어림해 줍니다. 1단계는 학습률 3개 × `EPOCHS` epoch 입니다. GPU 에 따라 수 시간이 걸릴 수 있으니, 샌티 체크의 스텝당 시간으로 먼저 어림해 보세요. 첫 epoch 의 val 결과가 명확히 나쁜 학습률은 중단해도 됩니다.

In [9]:
!pip install -q uv
!uv pip install --python /content/venv_llava/bin/python einops shortuuid ninja open-clip-torch "numpy<2"
!cd /content/LLaVA-Med && /content/venv_llava/bin/python -c "import torch, transformers, open_clip, llava; print('OK', torch.__version__, transformers.__version__)"


Using Python 3.10.12 environment at: /content/venv_llava
Resolved 52 packages in 13ms                                         
Uninstalled 4 packages in 158ms
Installed 30 packages in 206ms                              
 + cuda-bindings==13.4.4
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + einops==0.8.2
 + ftfy==6.3.1
 + ninja==1.13.2
 - numpy==2.2.6
 + numpy==1.26.4
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.24.0.43
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.30.7
 + nvidia-nvjitlink==13.5.37
 + nvidia-nvshmem-cu13==3.4.5
 + nvidia-nvtx==13.0.85
 + open-clip-torch==4.2.1
 + shortuuid==1.0.13
 + tiktoken==0.14.0
 + timm==1.0.31
 - torch==2.0.0
 + torch==2.14.1
 - torchvision==0.15.1
 + torchvision==0.29.2
 - triton==2.0.0
 + tri

In [2]:
# C-0a. 부록 C 단독 실행용 준비 — 앞 절을 실행하지 않았어도 된다 (이미 되어 있는 것은 건너뛴다)
import os, re, json, subprocess, glob, math
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_llavamed'      # 앞 절과 같은 폴더 (결과 파일이 모인다)
EVAL_DIR   = '/content/llava_eval_data'
PY         = '/content/venv_llava/bin/python'                  # 추론용 환경 (LLaVA-Med v1.0, torch 2.0.0)
for d in (DRIVE_ROOT, EVAL_DIR):
    os.makedirs(d, exist_ok=True)
def tee(cmd, log=None):
    # 명령의 출력을 한 줄씩 셀에 보여 주고(진행 상황), log 가 있으면 파일에도 저장한다. 종료 코드를 돌려준다.
    f = open(log, 'a') if log else None
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        if f:
            f.write(line); f.flush()
    rc = p.wait()
    if f:
        f.close()
    return rc

def sh(cmd):
    rc = tee(cmd)
    if rc != 0:
        raise RuntimeError(f'실패(종료 코드 {rc}): {cmd}')

# 1) 레포 (v1.0.0 태그)
if not os.path.isdir('/content/LLaVA-Med'):
    sh('git clone https://github.com/microsoft/LLaVA-Med.git /content/LLaVA-Med')
sh('cd /content/LLaVA-Med && git checkout v1.0.0')
os.chdir('/content/LLaVA-Med')

# 2) 추론용 venv — 파일이 있는지가 아니라 실제로 import 되는지로 점검하고, 안 되면 (재)설치한다
def venv_ok():
    if not os.path.exists(PY):
        return False, 'venv 없음'
    r = subprocess.run([PY, '-c', 'import torch, transformers, open_clip, einops, shortuuid, llava'],
                       capture_output=True, text=True, cwd='/content/LLaVA-Med')
    return r.returncode == 0, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ''

ok, why = venv_ok()
if not ok:
    print('추론 venv 점검 실패 →', why, '→ (재)설치합니다')
    sh('pip install -q uv')
    open('/content/constraints_llava.txt', 'w').write('torch==2.0.0\ntorchvision==0.15.1\n')   # 다른 패키지가 torch 를 올리지 못하게 고정
    if not os.path.exists(PY):
        sh('uv venv /content/venv_llava --python 3.10')
    CST = '-c /content/constraints_llava.txt'
    sh(f'uv pip install --python /content/venv_llava/bin/python {CST} torch==2.0.0 torchvision==0.15.1')
    sh(f'cd /content/LLaVA-Med && uv pip install --python /content/venv_llava/bin/python {CST} -e .')
    sh(f'uv pip install --python /content/venv_llava/bin/python {CST} einops shortuuid ninja open-clip-torch==2.24.0 "numpy<2"')
    ok, why = venv_ok()
    assert ok, f'설치 후에도 import 실패: {why}'
print(subprocess.run([PY, '-c', 'import torch, transformers, open_clip, llava; print(torch.__version__, transformers.__version__, torch.cuda.is_available())'],
                     capture_output=True, text=True, cwd='/content/LLaVA-Med').stdout.strip())

# 3) LLaMA-7B 베이스 (Drive 사본 우선, 없으면 HF)
from huggingface_hub import snapshot_download
LLAMA_BASE = '/content/drive/MyDrive/llama-7b-hf'
if not os.path.isdir(LLAMA_BASE):
    LLAMA_BASE = snapshot_download('huggyllama/llama-7b', local_dir='/content/llama-7b-hf', allow_patterns=['*.json', '*.model', '*.safetensors'])
assert os.path.exists(os.path.join(LLAMA_BASE, 'config.json')), 'LLaMA-7B 가중치가 올바르지 않습니다'
print('LLAMA_BASE =', LLAMA_BASE)

# 4) 추론 스크립트를 greedy 로 패치 (VQA-RAD 평가와 같은 조건)
p = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
s = open(p).read()
pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(pat, s):
    open(p, 'w').write(re.sub(pat, "do_sample=False,\n                max_new_tokens=1024,", s, count=1)); print('greedy 패치 완료')
else:
    assert 'do_sample=False,' in s, '패치 대상 문자열을 찾지 못했습니다'; print('greedy 패치 이미 적용됨')

ModuleNotFoundError: No module named 'google'

In [5]:
# C-0b. 공통 정의 — SLAKE 이미지, 질문 형식, 채점 함수 (기존 노트북과 같은 정의). test 문항 파일은 여기서 만들지 않는다 (C-9b).
import zipfile
from collections import Counter
import numpy as np
from huggingface_hub import hf_hub_download

SLAKE_ROOT = '/content/slake_data'
if not os.path.isdir(f'{SLAKE_ROOT}/imgs'):
    with zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')) as z:
        z.extractall(SLAKE_ROOT)
SLAKE_IMG_DIR = f'{SLAKE_ROOT}/imgs'

# closed 질문에 붙이는 접미어 (학습·평가 모두 동일)
CLOSED_SUFFIX = ' Please choose from the following two options: [yes, no]'

def write_eval_files(name, rows, get_image):
    """model_vqa_med.py 입력(JSON 배열 + conversations)과 채점용 참조 파일을 EVAL_DIR 에 만든다."""
    questions, refs = [], {}
    for i, r in enumerate(rows):
        questions.append({'id': i, 'image': get_image(r),
                          'conversations': [{'from': 'human', 'value': r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')},
                                            {'from': 'gpt', 'value': str(r['answer'])}]})
        refs[i] = {'answer': str(r['answer']), 'answer_type': r['answer_type']}
    json.dump(questions, open(f'{EVAL_DIR}/{name}_test.json', 'w'))
    json.dump(refs, open(f'{EVAL_DIR}/{name}_ref.json', 'w'))
    n_open = sum(v['answer_type'] == 'OPEN' for v in refs.values())
    print(f'{name}: {len(rows)}문항 (OPEN {n_open} / CLOSED {len(rows) - n_open})')

# 채점 (기존 방식): closed = 정답 단어가 예측 토큰에 포함되면 정답, open = 토큰 recall
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def score_jsonl(ref_file, pred_file):
    refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
    preds = {}
    for line in open(pred_file):
        p = json.loads(line)
        preds[int(p['question_id'])] = p['text']
    assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개 — 추론이 끝나지 않았거나 id 불일치'
    cl, op = [], []
    for i, ref in refs.items():
        if ref['answer_type'] == 'CLOSED':
            cl.append(closed_accuracy(preds[i], ref['answer']))
        else:
            op.append(open_recall(preds[i], ref['answer']))
    return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}
print('준비 완료')

준비 완료


In [6]:
# C-1. 출발점 모델: LLaVA-Med 의료 instruct 모델 (공식 SLAKE delta 는 받을 수 없음)
# apply_delta 는 LLaMA-7B(fp16 약 13.5GB)와 delta 를 동시에 메모리에 올린다 → 시스템 RAM 이 약 30GB 이상 필요하다 (Colab 고용량 RAM 런타임 권장).
# 저장 순서는 shard → index(pytorch_model.bin.index.json) → 토크나이저 이므로, 이 파일들이 모두 있고 shard 용량이 맞아야 "완전한 병합"이다.
import os, glob, shutil, json
from huggingface_hub import snapshot_download

INSTRUCT_DELTA = snapshot_download('microsoft/llava-med-7b-delta', local_dir='/content/delta_instruct')
INSTRUCT_MODEL = '/content/llava-med-7b-instruct'

def merged_ok(d):
    idx = f'{d}/pytorch_model.bin.index.json'
    if not os.path.exists(f'{d}/tokenizer.model') or not os.path.exists(f'{d}/config.json'):
        return False
    if os.path.exists(idx):
        meta = json.load(open(idx))
        files = sorted(set(meta['weight_map'].values()))
        if not all(os.path.exists(f'{d}/{f}') for f in files):
            return False
        total = sum(os.path.getsize(f'{d}/{f}') for f in files)
        return total >= 0.98 * meta['metadata']['total_size']
    return os.path.exists(f'{d}/pytorch_model.bin') and os.path.getsize(f'{d}/pytorch_model.bin') > 12e9

sh('free -g | head -2; df -h /content | tail -1')
if merged_ok(INSTRUCT_MODEL):
    print('이미 완전하게 병합돼 있습니다:', INSTRUCT_MODEL)
else:
    print('병합 결과가 없거나 불완전합니다 → 지우고 다시 병합합니다 (수 분 걸립니다. 중간에 셀을 중단하지 마세요)')
    shutil.rmtree(INSTRUCT_MODEL, ignore_errors=True)
    sh(f'cd /content/LLaVA-Med && {PY} -m llava.model.apply_delta --base-model-path {LLAMA_BASE} --target-model-path {INSTRUCT_MODEL} --delta-path {INSTRUCT_DELTA}')
assert merged_ok(INSTRUCT_MODEL), f'병합이 불완전합니다 ({INSTRUCT_MODEL}). 위 로그의 오류/Killed 여부를 확인하세요'
sh(f'ls -la {INSTRUCT_MODEL} | head -20')


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

               total        used        free      shared  buff/cache   available
Mem:              83           6          11           0          66          76
overlay         236G   72G  165G  31% /
이미 완전하게 병합돼 있습니다: /content/llava-med-7b-instruct
total 13169912
drwxr-xr-x 2 root root       4096 Nov  7 16:21 .
drwxr-xr-x 1 root root       4096 Nov  7 16:22 ..
-rw-r--r-- 1 root root         88 Nov  7 16:21 added_tokens.json
-rw-r--r-- 1 root root        788 Nov  7 16:20 config.json
-rw-r--r-- 1 root root        137 Nov  7 16:20 generation_config.json
-rw-r--r-- 1 root root 9976668264 Nov  7 16:21 pytorch_model-00001-of-00002.bin
-rw-r--r-- 1 root root 3508746633 Nov  7 16:21 pytorch_model-00002-of-00002.bin
-rw-r--r-- 1 root root      26924 Nov  7 16:21 pytorch_model.bin.index.json
-rw-r--r-- 1 root root         97 Nov  7 16:21 special_tokens_map.json
-rw-r--r-- 1 root root        833 Nov  7 16:21 tokenizer_config.json
-rw-r--r-- 1 root root     499723 Nov  7 16:21 tokenizer.model


In [7]:
# C-2. 학습용 별도 venv (기존 venv_llava 는 추론용으로 그대로 둔다)
# 중요: 다른 패키지가 torch 를 올려 버리면 bitsandbytes 0.41 이 CUDA 라이브러리를 못 찾는다 → 제약 파일로 torch/torchvision 을 고정한다.
!pip install -q uv
!rm -rf /content/venv_qlora
!uv venv /content/venv_qlora --python 3.10
QPY = '/content/venv_qlora/bin/python'
open('/content/constraints_qlora.txt', 'w').write('torch==2.0.0\ntorchvision==0.15.1\n')
!uv pip install --python {QPY} -c /content/constraints_qlora.txt torch==2.0.0 torchvision==0.15.1
!uv pip install --python {QPY} -c /content/constraints_qlora.txt transformers==4.31.0 tokenizers==0.13.3 accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.41.0 scipy psutil sentencepiece protobuf==3.20.3 einops shortuuid open-clip-torch==2.24.0 pillow matplotlib "numpy<2"
!cd /content/LLaVA-Med && uv pip install --python {QPY} -c /content/constraints_qlora.txt --no-deps -e .
# 점검: torch 의 CUDA 버전이 11.x 여야 하고, bitsandbytes 가 SUCCESS 를 보여야 한다
!{QPY} -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"
!{QPY} -m bitsandbytes 2>&1 | grep -E "SUCCESS|CUDA_VERSION|Setup Failed|Defaulting" | head
!cd /content/LLaVA-Med && {QPY} -c "import torch, transformers, peft, bitsandbytes, llava; import importlib.metadata as md; print('OK', torch.__version__, transformers.__version__, peft.__version__, md.version('bitsandbytes'))"


Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: /content/venv_qlora
Activate with: source /content/venv_qlora/bin/activate
Using Python 3.10.12 environment at: /content/venv_qlora
Resolved 33 packages in 173ms                                        
Installed 33 packages in 315ms                              
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + cmake==4.4.4
 + filelock==4.1.0
 + idna==3.20
 + jinja2==3.1.6
 + lit==23.1.5
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu11==11.10.3.66
 + nvidia-cuda-cupti-cu11==11.7.101
 + nvidia-cuda-nvrtc-cu11==11.7.99
 + nvidia-cuda-runtime-cu11==11.7.99
 + nvidia-cudnn-cu11==8.5.0.96
 + nvidia-cufft-cu11==10.9.0.58
 + nvidia-curand-cu11==10.2.10.91
 + nvidia-cusolver-cu11==11.4.0.1
 + nvidia-cusparse-cu11==11.7.4.91
 + nvidia-nccl-cu11==2.14.3
 + nvidia-nvtx-cu11==11.7.91
 + packaging==26.3
 + pillow==12.3.0
 + requests==2.34.2
 + setuptools==84.

In [8]:
# C-3. SLAKE 영어 train / val 생성 (질문 형식은 평가 파일과 동일: closed 에는 접미어). test 는 만들지 않는다.
import json, os
from huggingface_hub import hf_hub_download

QDATA = '/content/qlora_data'
os.makedirs(QDATA, exist_ok=True)

def build(src, out):
    rows = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', src, repo_type='dataset'))) if r['q_lang'] == 'en']
    res = []
    for r in rows:
        at = r['answer_type'].upper()
        res.append({'image': r['img_name'], 'question': r['question'] + (CLOSED_SUFFIX if at == 'CLOSED' else ''),
                    'answer': str(r['answer']), 'answer_type': at})
    json.dump(res, open(f'{QDATA}/{out}', 'w'))
    n_open = sum(x['answer_type'] == 'OPEN' for x in res)
    print(f'{src} -> {out}: {len(res)}문항 (OPEN {n_open} / CLOSED {len(res) - n_open})')
    assert all(os.path.exists(os.path.join(SLAKE_IMG_DIR, x['image'])) for x in res), '이미지 누락'
    return res

_tr, _va = build('train.json', 'slake_train.json'), build('validation.json', 'slake_val.json')
# train/val 겹침 점검 (이미지+질문). test 와의 점검은 test 를 처음 읽는 C-9b 에서 한다.
key = lambda r: (r['image'], r['question'])
print('train∩val:', len({key(r) for r in _tr} & {key(r) for r in _va}))
json.dump(_tr + _va, open(f'{QDATA}/slake_trainval.json', 'w'))
print('slake_trainval.json:', len(_tr) + len(_va), '문항 (train+val — 긴 학습에서 FINAL_TRAINVAL=True 일 때 사용)')


In [19]:
%%writefile /content/LLaVA-Med/train_qlora_slake.py
"""LLaVA-Med v1.0 (LLaMA-7B) 를 SLAKE 영어 train 으로 QLoRA 파인튜닝한다.
- 최적화에는 train 만 쓴다. val 은 매 epoch 평가(손실 + 생성 결과의 closed accuracy / open recall)해서 epoch·학습률 선택에만 쓴다. test 는 읽지 않는다.
- 프롬프트는 기존 평가 경로(llava/eval/model_vqa_med.py, conv_mode=simple)와 같은 형식이다.
  prompt  : system###Human: {질문}\n<im_start><im_patch>*256<im_end>###     (모델이 'Assistant: 답###' 을 생성)
  target  : 'Assistant: {정답}###'
- 실행 위치: /content/LLaVA-Med (llava 패키지가 import 되는 곳), venv_qlora 의 python 으로 실행.
"""
import argparse, json, os, re, glob, time
from collections import Counter

import torch
import transformers
from PIL import Image
from torch.utils.data import Dataset
from transformers import (AutoConfig, AutoTokenizer, BitsAndBytesConfig, CLIPImageProcessor, CLIPVisionModel,
                          Trainer, TrainingArguments, TrainerCallback)
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb
from llava import LlavaLlamaForCausalLM
from llava.conversation import conv_templates

IGNORE = -100
IM_PATCH, IM_START, IM_END = '<im_patch>', '<im_start>', '<im_end>'

ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True)            # 병합된 LLaVA-Med 7B (instruct) 경로
ap.add_argument('--train_json', required=True)
ap.add_argument('--val_json', required=True)
ap.add_argument('--image_folder', required=True)
ap.add_argument('--output_dir', required=True)
ap.add_argument('--lr', type=float, required=True)
ap.add_argument('--epochs', type=float, default=3)
ap.add_argument('--max_steps', type=int, default=-1)  # 샌티 체크용 (>0 이면 epochs 무시)
ap.add_argument('--limit_train', type=int, default=0)
ap.add_argument('--limit_val', type=int, default=0)
ap.add_argument('--bs', type=int, default=4)
ap.add_argument('--grad_accum', type=int, default=4)  # 유효 배치 16
ap.add_argument('--lora_r', type=int, default=64)
ap.add_argument('--lora_alpha', type=int, default=16)
ap.add_argument('--lora_dropout', type=float, default=0.05)
ap.add_argument('--max_len', type=int, default=1024)
ap.add_argument('--gen_bs', type=int, default=8)
ap.add_argument('--gen_new', type=int, default=24)
ap.add_argument('--logging_steps', type=int, default=10)
ap.add_argument('--seed', type=int, default=42)
ap.add_argument('--no_grad_ckpt', action='store_true')   # gradient checkpointing 을 끈다 (빠르지만 메모리를 더 쓴다)
ap.add_argument('--train_projector', action='store_true')   # mm_projector(이미지 특징→언어 모델 임베딩 변환층)도 함께 학습한다
ap.add_argument('--proj_lr', type=float, default=2e-5)      # mm_projector 학습률 (LoRA 는 --lr)
a = ap.parse_args()

os.makedirs(a.output_dir, exist_ok=True)
if os.path.exists(os.path.join(a.output_dir, 'run_info.json')):
    print('이미 끝난 실행입니다:', a.output_dir); raise SystemExit(0)
transformers.set_seed(a.seed)

# ───────── 채점 (기존 노트북과 같은 정의: closed accuracy / open recall, 참고로 open exact match) ─────────
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()

def closed_accuracy(pred, gt):
    return float(str(gt).strip().lower() in normalize(pred))

def open_recall(pred, gt):
    g, p = Counter(normalize(gt)), Counter(normalize(pred))
    return sum((g & p).values()) / sum(g.values()) if g else 0.0

def open_em(pred, gt):
    return float(normalize(pred) == normalize(gt))

def score(preds, rows):
    cl = [closed_accuracy(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'CLOSED']
    op = [open_recall(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'OPEN']
    em = [open_em(p, r['answer']) for p, r in zip(preds, rows) if r['answer_type'] == 'OPEN']
    m = lambda x: 100 * sum(x) / len(x) if x else float('nan')
    return {'closed': m(cl), 'open_recall': m(op), 'open_em': m(em), 'overall': m(cl + op), 'n_closed': len(cl), 'n_open': len(op)}

# ───────── 모델 (4비트) ─────────
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
cfg = AutoConfig.from_pretrained(a.model)
vt_name = getattr(cfg, 'mm_vision_tower', 'openai/clip-vit-large-patch14')
if hasattr(cfg, 'mm_vision_tower'):
    delattr(cfg, 'mm_vision_tower')   # 모델 생성자가 CLIP 을 중첩 로드하지 않게 한다 (device_map 로딩과 충돌 방지). 아래에서 직접 붙인다.
if not getattr(cfg, 'use_mm_proj', False):
    cfg.use_mm_proj, cfg.mm_hidden_size = True, 1024
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=dtype, llm_int8_skip_modules=['mm_projector', 'lm_head'])
model = LlavaLlamaForCausalLM.from_pretrained(a.model, config=cfg, quantization_config=bnb_cfg,
                                              torch_dtype=dtype, device_map={'': 0})
tok = AutoTokenizer.from_pretrained(a.model)
processor = CLIPImageProcessor.from_pretrained(vt_name)
vt = CLIPVisionModel.from_pretrained(vt_name, torch_dtype=dtype).cuda()
vt.requires_grad_(False)
model.model.vision_tower = [vt]
model.config.mm_vision_tower = vt_name
vc = vt.config
vc.im_patch_token = tok.convert_tokens_to_ids([IM_PATCH])[0]
vc.use_im_start_end = bool(getattr(model.config, 'mm_use_im_start_end', False))
vc.im_start_token, vc.im_end_token = tok.convert_tokens_to_ids([IM_START, IM_END])
assert vc.use_im_start_end, 'mm_use_im_start_end 가 꺼져 있습니다 (평가 경로와 형식이 달라집니다)'
assert tok.unk_token_id not in (vc.im_patch_token, vc.im_start_token, vc.im_end_token), '이미지 토큰이 토크나이저에 없습니다'
N_PATCH = (vc.image_size // vc.patch_size) ** 2
print('dtype', dtype, '| vision', vt_name, '| image tokens', N_PATCH)

model.config.use_cache = False
model.enable_input_require_grads()      # gradient checkpointing + LoRA: 임베딩 출력이 grad 를 요구하게 한다
if not a.no_grad_ckpt:
    model.gradient_checkpointing_enable()
print('gradient checkpointing:', not a.no_grad_ckpt)
targets = sorted({n.split('.')[-1] for n, m in model.named_modules()
                  if isinstance(m, bnb.nn.Linear4bit) and 'mm_projector' not in n and 'lm_head' not in n})
print('LoRA target modules:', targets)
assert targets, 'Linear4bit 모듈을 찾지 못했습니다 (4비트 로딩 확인)'
model = get_peft_model(model, LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=a.lora_dropout,
                                         target_modules=targets, bias='none', task_type='CAUSAL_LM',
                                         modules_to_save=['mm_projector'] if a.train_projector else None))
if a.train_projector:
    # modules_to_save 는 어댑터 체크포인트에 mm_projector 가중치를 함께 저장·복원하게 해 준다 (best epoch 복원 포함).
    # 작은 학습률에서 bf16 가중치 갱신이 반올림으로 사라지지 않도록 학습 대상 mm_projector 는 fp32 로 올린다 (순전파는 autocast).
    _p = [(n, p) for n, p in model.named_parameters() if 'mm_projector' in n and 'modules_to_save' in n]
    assert _p and all(p.requires_grad for _, p in _p), 'mm_projector 학습 파라미터를 찾지 못했습니다'
    for _, p in _p:
        p.data = p.data.float()
    print('mm_projector 학습:', [(n.split('mm_projector.')[-1], tuple(p.shape), p.dtype) for n, p in _p], '| proj_lr', a.proj_lr)
model.print_trainable_parameters()

# ───────── 데이터 ─────────
IM_IDS = tok.convert_tokens_to_ids([IM_START, IM_PATCH, IM_END])
MARK = '<<IMG>>'

def prompt_ids(r):
    """평가 경로와 같은 프롬프트의 토큰 id. 이 환경(transformers 4.31)의 토크나이저는 문자열 속 '<im_patch>' 를 한 토큰으로 인식하지 못하고
    여러 조각으로 쪼개므로(시퀀스가 ~1100 토큰이 됨), 이미지 토큰 id 는 직접 끼워 넣는다. 앞·뒤 글자만 토크나이저로 변환한다."""
    conv = conv_templates['simple'].copy()
    conv.append_message(conv.roles[0], r['question'] + '\n' + MARK)
    pre, post = conv.get_prompt().split(MARK)
    return tok(pre).input_ids + [IM_IDS[0]] + [IM_IDS[1]] * N_PATCH + [IM_IDS[2]] + tok(post, add_special_tokens=False).input_ids

def load_pixel(r):
    img = Image.open(os.path.join(a.image_folder, r['image'])).convert('RGB')
    return processor.preprocess(img, return_tensors='pt')['pixel_values'][0]

class QADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        p_ids = prompt_ids(r)
        a_ids = tok('Assistant: ' + r['answer'] + '###', add_special_tokens=False).input_ids
        ids, labels = p_ids + a_ids, [IGNORE] * len(p_ids) + a_ids
        assert len(ids) <= a.max_len, f'시퀀스가 max_len 을 넘습니다: {len(ids)}'
        return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels), 'image': load_pixel(r).to(dtype)}

def collate(batch):
    ids = torch.nn.utils.rnn.pad_sequence([b['input_ids'] for b in batch], batch_first=True, padding_value=tok.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence([b['labels'] for b in batch], batch_first=True, padding_value=IGNORE)
    return {'input_ids': ids, 'labels': labels, 'attention_mask': ids.ne(tok.pad_token_id),
            'images': torch.stack([b['image'] for b in batch])}

train_rows = json.load(open(a.train_json)); val_rows = json.load(open(a.val_json))
if a.limit_train: train_rows = train_rows[:a.limit_train]
if a.limit_val and a.limit_val < len(val_rows):
    import random
    val_rows = random.Random(a.seed).sample(val_rows, a.limit_val)   # 시드 고정 무작위 부분집합 (모든 학습률이 같은 문항으로 평가)
print(f'train {len(train_rows)} / val {len(val_rows)}')
_ex = prompt_ids(train_rows[0])
print('예시 프롬프트 길이', len(_ex), '| 이미지 토큰 수', sum(t == IM_IDS[1] for t in _ex), '| 앞 8토큰', tok.convert_ids_to_tokens(_ex[:8]))
assert sum(t == IM_IDS[1] for t in _ex) == N_PATCH and len(_ex) < 700, '프롬프트 토큰 구성이 예상과 다릅니다'

# ───────── val 생성 평가 (매 epoch) ─────────
@torch.no_grad()
def generate_val(m):
    m.eval()
    preds = []
    for i in range(0, len(val_rows), a.gen_bs):
        chunk = val_rows[i:i + a.gen_bs]
        seqs = [prompt_ids(r) for r in chunk]
        L = max(len(s) for s in seqs)
        ids = torch.tensor([[tok.pad_token_id] * (L - len(s)) + s for s in seqs]).cuda()          # 왼쪽 패딩
        att = torch.tensor([[0] * (L - len(s)) + [1] * len(s) for s in seqs]).cuda()
        imgs = torch.stack([load_pixel(r) for r in chunk]).to(device='cuda', dtype=dtype)
        with torch.autocast('cuda', dtype=dtype):      # fp32 mm_projector 와 bf16 특징이 섞여도 되게 한다
            out = m.generate(input_ids=ids, attention_mask=att, images=imgs, do_sample=False, max_new_tokens=a.gen_new, use_cache=True)
        for t in tok.batch_decode(out[:, L:], skip_special_tokens=True):
            t = t.split('###')[0].strip()
            preds.append(re.sub(r'^Assistant:\s*', '', t).strip())
    return preds

class ValGen(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, metrics=None, **kw):
        t0 = time.time()
        preds = generate_val(model)
        s = score(preds, val_rows)
        metrics.update({'eval_val_closed': s['closed'], 'eval_val_open_recall': s['open_recall'],
                        'eval_val_open_em': s['open_em'], 'eval_val_overall': s['overall']})   # metric_for_best_model 이 읽는다
        ep = float(state.epoch or 0)
        recent = [h['loss'] for h in state.log_history if 'loss' in h and h.get('epoch', 0) > ep - 1.0]
        rec = {'lr': a.lr, 'epoch': round(ep, 3), 'step': state.global_step,
               'train_loss': sum(recent) / len(recent) if recent else None, 'eval_loss': metrics.get('eval_loss'), **s,
               'gen_sec': round(time.time() - t0)}
        with open(os.path.join(a.output_dir, 'val_metrics.jsonl'), 'a') as f:
            f.write(json.dumps(rec) + '\n')
        json.dump([{'pred': p, 'answer': r['answer'], 'answer_type': r['answer_type']} for p, r in zip(preds, val_rows)],
                  open(os.path.join(a.output_dir, f'val_pred_step{state.global_step}.json'), 'w'))
        print('[VAL]', json.dumps(rec), flush=True)

# ───────── 학습 ─────────
sanity = a.max_steps > 0
strat = dict(evaluation_strategy='steps', eval_steps=a.max_steps, save_strategy='steps', save_steps=a.max_steps) if sanity \
    else dict(evaluation_strategy='epoch', save_strategy='epoch')
args = TrainingArguments(
    output_dir=a.output_dir, per_device_train_batch_size=a.bs, per_device_eval_batch_size=a.bs,
    gradient_accumulation_steps=a.grad_accum, num_train_epochs=a.epochs, max_steps=a.max_steps,
    learning_rate=a.lr, lr_scheduler_type='cosine', warmup_ratio=0.03, weight_decay=0.0, max_grad_norm=0.3,
    bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16), optim='paged_adamw_32bit',
    logging_steps=a.logging_steps, save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='val_overall', greater_is_better=True, report_to='none',
    remove_unused_columns=False, label_names=['labels'], dataloader_num_workers=2, seed=a.seed, **strat)
class ProjLRTrainer(Trainer):
    """LoRA 와 mm_projector 에 서로 다른 학습률을 주는 옵티마이저 (스케줄러는 각 그룹의 기준 학습률에 같은 비율을 곱한다)."""
    def create_optimizer(self):
        if self.optimizer is None:
            proj = [p for n, p in self.model.named_parameters() if p.requires_grad and 'mm_projector' in n]
            rest = [p for n, p in self.model.named_parameters() if p.requires_grad and 'mm_projector' not in n]
            cls, kw = Trainer.get_optimizer_cls_and_kwargs(self.args)
            kw.pop('lr', None)
            self.optimizer = cls([{'params': rest, 'lr': self.args.learning_rate}, {'params': proj, 'lr': a.proj_lr}], **kw)
        return self.optimizer

trainer = (ProjLRTrainer if a.train_projector else Trainer)(model=model, args=args, train_dataset=QADataset(train_rows), eval_dataset=QADataset(val_rows),
                                                              data_collator=collate, callbacks=[ValGen()])
resume = bool(glob.glob(os.path.join(a.output_dir, 'checkpoint-*')))
print('resume_from_checkpoint =', resume)
trainer.train(resume_from_checkpoint=resume or None)

best = trainer.state.best_model_checkpoint
ckpts = sorted(glob.glob(os.path.join(a.output_dir, 'checkpoint-*')), key=lambda d: int(d.rsplit('-', 1)[1]))
info = {'lr': a.lr, 'epochs': a.epochs, 'max_steps': a.max_steps, 'best_checkpoint': best, 'last_checkpoint': ckpts[-1] if ckpts else None,
        'best_val_overall': trainer.state.best_metric, 'n_train': len(train_rows), 'n_val': len(val_rows),
        'train_projector': a.train_projector, 'proj_lr': a.proj_lr if a.train_projector else None,
        'grad_ckpt': not a.no_grad_ckpt, 'peak_gpu_mem_gb': round(torch.cuda.max_memory_allocated() / 2**30, 1)}
json.dump(trainer.state.log_history, open(os.path.join(a.output_dir, 'log_history.json'), 'w'))
json.dump(info, open(os.path.join(a.output_dir, 'run_info.json'), 'w'), indent=1)
print('DONE', json.dumps(info))


Overwriting /content/LLaVA-Med/train_qlora_slake.py


In [18]:
%%writefile /content/LLaVA-Med/merge_qlora.py
"""QLoRA 어댑터를 fp16 LLaVA-Med 위에 병합해 기존 추론 경로(model_vqa_med.py)가 읽는 HF 체크포인트로 저장한다.
참고: 어댑터는 4비트 양자화된 기반 위에서 학습했고 병합은 fp16 기반 위에서 하므로 양자화 오차만큼 출력이 미세하게 달라질 수 있다.
"""
import argparse, os, torch
from transformers import AutoConfig, AutoTokenizer
from peft import PeftModel
from llava import LlavaLlamaForCausalLM

ap = argparse.ArgumentParser()
ap.add_argument('--base', required=True)      # 병합된 LLaVA-Med 7B (instruct)
ap.add_argument('--adapter', required=True)   # Trainer 체크포인트 폴더 (adapter_model.bin 이 있는 곳)
ap.add_argument('--out', required=True)
a = ap.parse_args()

assert os.path.exists(os.path.join(a.adapter, 'adapter_config.json')), f'어댑터가 없습니다: {a.adapter}'
cfg = AutoConfig.from_pretrained(a.base)
vt_name = getattr(cfg, 'mm_vision_tower', 'openai/clip-vit-large-patch14')
if hasattr(cfg, 'mm_vision_tower'):
    delattr(cfg, 'mm_vision_tower')
model = LlavaLlamaForCausalLM.from_pretrained(a.base, config=cfg, torch_dtype=torch.float16, device_map={'': 0})
model = PeftModel.from_pretrained(model, a.adapter).merge_and_unload()

# mm_projector 를 함께 학습한 어댑터라면(modules_to_save) 그 가중치를 확실히 반영한다 (peft 의 병합 동작에 기대지 않고 직접 읽어서 넣는다)
sd = None
for fn in ('adapter_model.bin', 'adapter_model.safetensors'):
    pth = os.path.join(a.adapter, fn)
    if os.path.exists(pth):
        if fn.endswith('.bin'):
            sd = torch.load(pth, map_location='cpu')
        else:
            from safetensors.torch import load_file
            sd = load_file(pth)
        break
proj = {k.split('mm_projector.')[-1].replace('modules_to_save.default.', ''): v for k, v in (sd or {}).items()
        if 'mm_projector' in k and 'modules_to_save' in k}
if proj:
    old = model.model.mm_projector
    new = torch.nn.Linear(proj['weight'].shape[1], proj['weight'].shape[0])
    new.load_state_dict({k: v.float() for k, v in proj.items()})
    new = new.half().to(next(model.parameters()).device)
    print('mm_projector 반영: 병합 전과의 최대 차이', float((new.weight - old.weight.to(new.weight.dtype)).abs().max()) if hasattr(old, 'weight') else '(래퍼)')
    model.model.mm_projector = new
else:
    print('어댑터에 mm_projector 가중치가 없습니다 (LoRA 만 학습한 어댑터)')
model.config.mm_vision_tower = vt_name
model.config.use_cache = True
os.makedirs(a.out, exist_ok=True)
model.save_pretrained(a.out, safe_serialization=False, max_shard_size='5GB')
# 토크나이저는 다시 저장하지 않고 기반 폴더의 파일을 그대로 복사한다 (이 환경의 transformers 4.31 로 다시 저장하면 추가 토큰 정보가 달라질 수 있다)
import shutil
for f in ('tokenizer.model', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json'):
    if os.path.exists(os.path.join(a.base, f)):
        shutil.copy(os.path.join(a.base, f), os.path.join(a.out, f))
print('merged ->', a.out)


Overwriting /content/LLaVA-Med/merge_qlora.py


In [5]:
# C-4. 설정 (학습률만 다르게, 나머지는 train_qlora_slake.py 의 기본값으로 동일)
import os, json, subprocess, glob, math
QROOT = f'{DRIVE_ROOT}/qlora_slake'
LRS = [2e-5, 1e-4, 2e-4]
EPOCHS = 2          # 1단계: 학습률 3개를 비교하는 학습 길이 (비용 절감: 학습률마다 이 길이로만 돈다)
VAL_LIMIT_SEARCH = 400   # 1단계에서 epoch 마다 생성 평가할 val 문항 수 (시드 고정 무작위 부분집합, 0=전체). 2단계는 항상 val 전체
FINAL_EPOCHS = 9    # 2단계: val 로 선택된 학습률 하나만 이 길이로 다시 학습 (공식 SLAKE delta 와 같은 9 epoch). 생략해도 된다 (C-9c/9d)
FINAL_TRAINVAL = True    # 2단계를 train+val 로 학습 (규칙 일부 완화 — 결과표에 별도 표기). val 선택이 없으므로 마지막 epoch 를 보고한다. False 면 train 만 학습하고 val 로 선택
FINAL_VAL_LIMIT = 100    # FINAL_TRAINVAL 일 때 epoch 마다 생성 평가할 val 문항 수 (val 이 학습에 들어가 진단용일 뿐이므로 적게)

TRAIN_PROJECTOR = True   # 2단계에서 mm_projector(이미지 특징→언어 모델 변환층)도 LoRA 와 함께 학습 (1단계 학습률 탐색은 LoRA 만 학습했다)
PROJ_LR = 2e-5           # mm_projector 학습률 (LoRA 는 1단계에서 val 로 고른 값). LLaVA 의 LoRA 학습 방식과 같은 값
FINAL_STAGE = 'final_proj' if TRAIN_PROJECTOR else 'final'   # 설정이 다르면 2단계 결과 폴더도 분리
if FINAL_TRAINVAL:
    FINAL_STAGE += '_trainval'
GRAD_CKPT = False   # False: gradient checkpointing 끔 (빠름, 메모리 더 씀). GPU 메모리 부족(OOM)이면 True 로
SANITY_STAGE = 'sanity_gc' if GRAD_CKPT else 'sanity_nogc'   # 설정이 다르면 샌티 체크 결과 폴더도 분리

def lr_dir(stage, lr):
    return f'{QROOT}/{stage}/lr{lr:g}'

def train_cmd(lr, stage, extra=''):
    proj = f'--train_projector --proj_lr {PROJ_LR}' if (TRAIN_PROJECTOR and stage == FINAL_STAGE) else ''
    tv = FINAL_TRAINVAL and stage == FINAL_STAGE
    tj = f'{QDATA}/slake_trainval.json' if tv else f'{QDATA}/slake_train.json'
    if tv:
        extra += f' --limit_val {FINAL_VAL_LIMIT}'
    return (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {QPY} train_qlora_slake.py --model {INSTRUCT_MODEL} '
            f'--train_json {tj} --val_json {QDATA}/slake_val.json --image_folder {SLAKE_IMG_DIR} '
            f'--output_dir {lr_dir(stage, lr)} --lr {lr} {"" if GRAD_CKPT else "--no_grad_ckpt"} {proj} {extra}')
print({lr: lr_dir('full', lr) for lr in LRS})

In [2]:
# C-5. 샌티 체크: 학습률마다 40스텝 (유효 배치 16 × 40 = 640문항). 발산/불안정/오류 확인용 — 이 결과로 학습률을 고르지 않는다.
SANITY_STEPS = 40
for lr in LRS:
    d = lr_dir(SANITY_STAGE, lr)
    os.makedirs(d, exist_ok=True)
    cmd = train_cmd(lr, SANITY_STAGE, f'--max_steps {SANITY_STEPS} --limit_train {SANITY_STEPS * 16} --limit_val 64 --logging_steps 5')
    open(f'{d}/stdout.log', 'w').close()
    print(f'===== lr={lr:g} =====')
    rc = tee(cmd, f'{d}/stdout.log')
    print(f'lr={lr:g}  종료 코드 {rc}  (로그: {d}/stdout.log)')

NameError: name 'LRS' is not defined

In [35]:
# C-6. 샌티 체크 결과 (손실 곡선, NaN/발산 여부, val 점검값). 오류가 났으면 stdout.log 의 끝부분을 보여 준다.
for lr in LRS:
    d = lr_dir(SANITY_STAGE, lr)
    try:
        hist = [x for x in json.load(open(f'{d}/log_history.json')) if 'loss' in x]
    except FileNotFoundError:
        print(f'lr={lr:g}: 결과 없음 → 로그 끝부분'); print(''.join(open(f'{d}/stdout.log', errors='ignore').readlines()[-25:])); continue
    ls = [x['loss'] for x in hist]
    bad = any(math.isnan(l) or math.isinf(l) for l in ls)
    print(f'lr={lr:g}  train loss: ' + ' '.join(f'{l:.3f}' for l in ls) + ('   !! NaN/inf' if bad else '') + ('   ! 마지막 손실이 처음보다 큼' if ls[-1] > ls[0] else ''))
    for l in open(f'{d}/val_metrics.jsonl'):
        v = json.loads(l); print(f'   val(64문항) closed {v["closed"]:.1f} / open recall {v["open_recall"]:.1f} / eval_loss {v["eval_loss"]:.3f}  (생성 {v["gen_sec"]}초)')

# ── 비용 어림: 샌티 체크의 처리 속도로 전체 시간을 계산한다 (GPU/상태에 따라 달라지는 대략값) ──
try:
    d0 = lr_dir(SANITY_STAGE, LRS[0])
    fin = [x for x in json.load(open(f'{d0}/log_history.json')) if 'train_samples_per_second' in x][-1]
    sps = fin['train_samples_per_second']
    gen_per = json.loads(open(f'{d0}/val_metrics.jsonl').readline())['gen_sec'] / 64          # val 1문항 생성 시간(초)
    n_srch = VAL_LIMIT_SEARCH or len(_va)
    m_srch = (len(_tr) / sps + gen_per * n_srch) / 60       # 1단계 1 epoch (분)
    m_final = (len(_tr) / sps + gen_per * len(_va)) / 60    # 2단계 1 epoch (분)
    print(f'\n처리 속도 {sps:.2f} 문항/초 | train {len(_tr)}문항 → 학습만 1 epoch ≈ {len(_tr) / sps / 60:.1f}분 | val 생성 {gen_per:.2f}초/문항')
    print(f'  1단계: 1 epoch ≈ {m_srch:.1f}분 (val {n_srch}문항 평가 포함) × 3학습률 × {EPOCHS} epoch ≈ {3 * EPOCHS * m_srch / 60:.1f}시간')
    print(f'  2단계: 1 epoch ≈ {m_final:.1f}분 (val {len(_va)}문항) × {FINAL_EPOCHS} epoch ≈ {FINAL_EPOCHS * m_final / 60:.1f}시간')
    print(f'  합계 ≈ {(3 * EPOCHS * m_srch + FINAL_EPOCHS * m_final) / 60:.1f}시간   (참고: 3학습률 × 9 epoch 전부 = {27 * m_final / 60:.1f}시간)')
except Exception as e:
    print('비용 어림을 계산하지 못했습니다:', e)


lr=2e-05  train loss: 4.894 4.803 4.645 4.437 4.013 4.041 3.748 3.787
   val(64문항) closed 36.0 / open recall 34.9 / eval_loss 3.706  (생성 39초)
lr=0.0001  train loss: 4.776 3.322 1.766 0.737 0.512 0.431 0.242 0.311
   val(64문항) closed 56.0 / open recall 18.4 / eval_loss 0.878  (생성 39초)
lr=0.0002  train loss: 4.561 1.679 0.639 0.249 0.301 0.239 0.129 0.148
   val(64문항) closed 68.0 / open recall 29.9 / eval_loss 0.732  (생성 39초)

처리 속도 4.52 문항/초 | train 4919문항 → 학습만 1 epoch ≈ 18.1분 | val 생성 0.61초/문항
  1단계: 1 epoch ≈ 22.2분 (val 400문항 평가 포함) × 3학습률 × 2 epoch ≈ 2.2시간
  2단계: 1 epoch ≈ 28.8분 (val 1053문항) × 6 epoch ≈ 2.9시간
  합계 ≈ 5.1시간   (참고: 3학습률 × 9 epoch 전부 = 13.0시간)


In [36]:
# C-7. 1단계(학습률 탐색): 학습률 3개를 차례로 EPOCHS epoch 씩 (백그라운드). 끊겨도 이 셀을 다시 실행하면 체크포인트에서 이어 가고, 끝난 실행은 건너뛴다.
if 'full_proc' in globals() and full_proc.poll() is None:
    print('이미 실행 중입니다. PID', full_proc.pid)
else:
    for lr in LRS:
        os.makedirs(lr_dir('full', lr), exist_ok=True)
    cmds = ' ; '.join(train_cmd(lr, 'full', f'--epochs {EPOCHS} --limit_val {VAL_LIMIT_SEARCH}') + f' >> {lr_dir("full", lr)}/stdout.log 2>&1' for lr in LRS)
    full_proc = subprocess.Popen(['bash', '-c', cmds], start_new_session=True)
    print('PID', full_proc.pid)

PID 15973


In [134]:
# C-8. 진행 확인 (epoch 마다 val 손실과 val 생성 점수가 쌓인다)
print('프로세스 종료 코드(None=실행 중):', full_proc.poll())
for lr in LRS:
    p = f'{lr_dir("full", lr)}/val_metrics.jsonl'
    print(f'--- lr={lr:g}')
    if os.path.exists(p):
        for l in open(p):
            v = json.loads(l)
            print(f'  epoch {v["epoch"]:.2f}  train_loss {v["train_loss"]:.4f}  val_loss {v["eval_loss"]:.4f}  closed {v["closed"]:.2f}  open recall {v["open_recall"]:.2f}  open EM {v["open_em"]:.2f}  overall {v["overall"]:.2f}')
    print('  로그 끝:', ''.join(open(f'{lr_dir("full", lr)}/stdout.log', errors='ignore').readlines()[-2:]).strip()[:200] if os.path.exists(f'{lr_dir("full", lr)}/stdout.log') else '(없음)')

프로세스 종료 코드(None=실행 중): 0
--- lr=2e-05
  epoch 1.00  train_loss 1.0239  val_loss 0.2637  closed 60.26  open recall 66.36  open EM 61.48  overall 63.98
  epoch 2.00  train_loss 0.2063  val_loss 0.2151  closed 61.54  open recall 69.92  open EM 64.75  overall 66.65
  로그 끝: 100%|██████████| 614/614 [32:10<00:00,  3.14s/it]
DONE {"lr": 2e-05, "epochs": 2.0, "max_steps": -1, "best_checkpoint": "/content/drive/MyDrive/expert_cfg_llavamed/qlora_slake/full/lr2e-05/checkpoint-
--- lr=0.0001
  epoch 1.00  train_loss 0.5124  val_loss 0.1274  closed 58.97  open recall 74.61  open EM 68.85  overall 68.51
  epoch 2.00  train_loss 0.1114  val_loss 0.1005  closed 63.46  open recall 80.85  open EM 75.00  overall 74.07
  로그 끝: 100%|██████████| 614/614 [31:59<00:00,  3.13s/it]
DONE {"lr": 0.0001, "epochs": 2.0, "max_steps": -1, "best_checkpoint": "/content/drive/MyDrive/expert_cfg_llavamed/qlora_slake/full/lr0.0001/checkpoin
--- lr=0.0002
  epoch 1.00  train_loss 0.4254  val_loss 0.1166  closed 58.97  open

In [4]:
# C-9. 선택: train 손실이 아니라 val 성능(overall, 동점이면 val 손실)으로 학습률·epoch 를 고른다. test 는 쓰지 않는다.
import numpy as np
import matplotlib.pyplot as plt

runs = {}
for lr in LRS:
    p = f'{lr_dir("full", lr)}/val_metrics.jsonl'
    if os.path.exists(p):
        runs[lr] = [json.loads(l) for l in open(p)]
assert runs, 'val_metrics.jsonl 이 없습니다 (본 실험 결과 필요)'

def best_of(recs):
    return max(recs, key=lambda v: (v['overall'], -v['eval_loss']))

print(f"{'LR':>8s} {'Epochs':>6s} {'Val-selected epoch':>19s} {'Val open':>9s} {'Val closed':>10s} {'Val overall':>11s} {'Train loss':>10s} {'Val loss':>9s} {'Last-epoch val overall':>23s}")
sel_lr, sel_rec = None, None
for lr, recs in runs.items():
    b, last = best_of(recs), recs[-1]
    print(f"{lr:8g} {int(round(last['epoch'])):6d} {int(round(b['epoch'])):19d} {b['open_recall']:9.2f} {b['closed']:10.2f} {b['overall']:11.2f} {b['train_loss']:10.4f} {b['eval_loss']:9.4f} {last['overall']:23.2f}")
    if sel_rec is None or (b['overall'], -b['eval_loss']) > (sel_rec['overall'], -sel_rec['eval_loss']):
        sel_lr, sel_rec = lr, b
info = json.load(open(f'{lr_dir("full", sel_lr)}/run_info.json'))
SEL = {'lr': sel_lr, 'epoch': sel_rec['epoch'], 'best_checkpoint': info['best_checkpoint'], 'last_checkpoint': info['last_checkpoint'], 'val': sel_rec}
print(f"\n선택: lr={sel_lr:g}, epoch {sel_rec['epoch']:.0f} (val overall {sel_rec['overall']:.2f})  ← train 손실이 아니라 val 성능 기준")
print('best ckpt:', SEL['best_checkpoint'], '| last ckpt:', SEL['last_checkpoint'])

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for lr, recs in runs.items():
    ep = [v['epoch'] for v in recs]
    ax[0, 0].plot(ep, [v['train_loss'] for v in recs], 'o-', label=f'{lr:g}')
    ax[0, 1].plot(ep, [v['eval_loss'] for v in recs], 'o-', label=f'{lr:g}')
    ax[1, 0].plot(ep, [v['overall'] for v in recs], 'o-', label=f'{lr:g}')
    l, = ax[1, 1].plot(ep, [v['closed'] for v in recs], 'o-', label=f'closed {lr:g}')
    ax[1, 1].plot(ep, [v['open_recall'] for v in recs], 's--', color=l.get_color(), label=f'open recall {lr:g}')
for a_, t in zip(ax.ravel(), ['(a) train loss', '(b) val loss', '(c) val overall per epoch', '(d) val closed (solid) / open recall (dashed)']):
    a_.set_title(t); a_.set_xlabel('epoch'); a_.legend(fontsize=7)
ax[0, 0].set_yscale('log')
fig.suptitle('SLAKE QLoRA (LLaVA-Med v1.0) — learning-rate comparison, selected on validation only')
plt.tight_layout(); plt.savefig(f'{QROOT}/val_curves.png', dpi=150); plt.show()

NameError: name 'LRS' is not defined

In [137]:
# C-9a. (진단) 선택된 학습률의 val 생성 결과로 closed 오답 유형을 본다 — closed 정확도가 논문(85)보다 많이 낮은 이유를 2단계 전에 확인. GPU 불필요, test 미사용.
import glob, collections
assert 'SEL' in globals(), '먼저 C-9 로 학습률을 선택하세요'
d = SEL.get('dir', lr_dir('full', SEL['lr']))      # 2단계를 했으면 그 폴더, 아니면 1단계 폴더
f = sorted(glob.glob(f'{d}/val_pred_step*.json'), key=lambda p: int(p.rsplit('step', 1)[1].split('.')[0]))[-1]
rows = json.load(open(f))
print('파일:', os.path.basename(f), '| 문항', len(rows))

cl = [r for r in rows if r['answer_type'] == 'CLOSED']
def lab(p):
    t = normalize(p)
    return 'yes' if ('yes' in t and 'no' not in t) else ('no' if ('no' in t and 'yes' not in t) else 'other')
gt = collections.Counter(r['answer'].strip().lower() for r in cl)
print(f'closed {len(cl)}문항 | 정답 분포 {dict(gt)} | 다수 클래스 기준선 {100 * max(gt.values()) / len(cl):.1f}%')
print('예측 분포:', dict(collections.Counter(lab(r['pred']) for r in cl)))
print('(정답, 예측) 혼동:', dict(collections.Counter((r['answer'].strip().lower(), lab(r['pred'])) for r in cl)))
print('closed 오답 예시:')
for r in [r for r in cl if closed_accuracy(r['pred'], r['answer']) == 0][:8]:
    print(f"   정답 {r['answer']!r:8} | 예측 {r['pred'][:70]!r}")
op = [r for r in rows if r['answer_type'] == 'OPEN']
print('open 예시 (정답 → 예측):')
for r in op[:6]:
    print(f"   {r['answer'][:30]!r:34} → {r['pred'][:40]!r}")

파일: val_pred_step614.json | 문항 400
closed 156문항 | 정답 분포 {'yes': 70, 'no': 69, 'coronal plane': 1, 'liver': 3, 'lung': 3, 'transverse plane': 3, 'right': 2, 'lower right': 1, 'heart': 2, 't1': 2} | 다수 클래스 기준선 44.9%
예측 분포: {'no': 98, 'yes': 41, 'other': 17}
(정답, 예측) 혼동: {('yes', 'no'): 39, ('no', 'no'): 59, ('yes', 'yes'): 31, ('coronal plane', 'other'): 1, ('liver', 'other'): 3, ('lung', 'other'): 3, ('no', 'yes'): 10, ('transverse plane', 'other'): 3, ('right', 'other'): 2, ('lower right', 'other'): 1, ('heart', 'other'): 2, ('t1', 'other'): 2}
closed 오답 예시:
   정답 'Yes'    | 예측 'No'
   정답 'Yes'    | 예측 'No'
   정답 'Yes'    | 예측 'No'
   정답 'Coronal Plane' | 예측 'Coronal Plane'
   정답 'Yes'    | 예측 'No'
   정답 'No'     | 예측 'Yes'
   정답 'Yes'    | 예측 'No'
   정답 'Transverse Plane' | 예측 'Transverse Plane'
open 예시 (정답 → 예측):
   'Lung'                             → 'Lung'
   'Black'                            → 'Black'
   'CT'                               → 'CT'
   'Atelectasis'                 

In [3]:
%%writefile /content/LLaVA-Med/ablate_val.py
"""진단: QLoRA 어댑터(또는 출발점 모델)가 질문에 답할 때 이미지를 실제로 쓰는지 확인한다 (val 만 사용, test 는 읽지 않는다).
같은 질문에 (a) 맞는 이미지 (b) 다른 문항의 이미지를 넣어 점수를 비교한다. 두 값이 비슷하면 모델이 이미지를 거의 보지 않는 것이다.
  --kind yn   : closed 중 정답이 yes/no 인 문항, 정확도
  --kind open : open 문항, 토큰 recall (기존 채점 방식)
실행 위치: /content/LLaVA-Med, venv_qlora 의 python.
"""
import argparse, json, os, re, random
from collections import Counter

import torch
from PIL import Image
from transformers import AutoConfig, AutoTokenizer, BitsAndBytesConfig, CLIPImageProcessor, CLIPVisionModel
from peft import PeftModel
from llava import LlavaLlamaForCausalLM
from llava.conversation import conv_templates

IM_PATCH, IM_START, IM_END = '<im_patch>', '<im_start>', '<im_end>'
ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True)
ap.add_argument('--adapter', default='')            # 비우면 어댑터 없이(= 출발점 instruct 모델) 평가
ap.add_argument('--val_json', required=True)
ap.add_argument('--image_folder', required=True)
ap.add_argument('--kind', choices=['yn', 'open'], default='yn')
ap.add_argument('--n', type=int, default=0)         # 0 = yn 은 전부, open 은 300문항(시드 고정 무작위)
ap.add_argument('--gen_bs', type=int, default=8)
ap.add_argument('--gen_new', type=int, default=24)
ap.add_argument('--seed', type=int, default=42)
a = ap.parse_args()

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
cfg = AutoConfig.from_pretrained(a.model)
vt_name = getattr(cfg, 'mm_vision_tower', 'openai/clip-vit-large-patch14')
if hasattr(cfg, 'mm_vision_tower'):
    delattr(cfg, 'mm_vision_tower')
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=dtype, llm_int8_skip_modules=['mm_projector', 'lm_head'])
model = LlavaLlamaForCausalLM.from_pretrained(a.model, config=cfg, quantization_config=bnb_cfg, torch_dtype=dtype, device_map={'': 0})
tok = AutoTokenizer.from_pretrained(a.model)
processor = CLIPImageProcessor.from_pretrained(vt_name)
vt = CLIPVisionModel.from_pretrained(vt_name, torch_dtype=dtype).cuda()
vt.requires_grad_(False)
model.model.vision_tower = [vt]
model.config.mm_vision_tower = vt_name
vc = vt.config
vc.im_patch_token = tok.convert_tokens_to_ids([IM_PATCH])[0]
vc.use_im_start_end = True
vc.im_start_token, vc.im_end_token = tok.convert_tokens_to_ids([IM_START, IM_END])
N_PATCH = (vc.image_size // vc.patch_size) ** 2
IM_IDS = tok.convert_tokens_to_ids([IM_START, IM_PATCH, IM_END])
if a.adapter:
    model = PeftModel.from_pretrained(model, a.adapter)
    print('adapter:', a.adapter)
else:
    print('어댑터 없음 (출발점 instruct 모델)')
model.eval()

def prompt_ids(q):
    conv = conv_templates['simple'].copy()
    conv.append_message(conv.roles[0], q + '\n<<IMG>>')
    pre, post = conv.get_prompt().split('<<IMG>>')
    return tok(pre).input_ids + [IM_IDS[0]] + [IM_IDS[1]] * N_PATCH + [IM_IDS[2]] + tok(post, add_special_tokens=False).input_ids

def pixel(name):
    return processor.preprocess(Image.open(os.path.join(a.image_folder, name)).convert('RGB'), return_tensors='pt')['pixel_values'][0]

def toks(text):
    return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower().replace('###', ' ')).split()

def yn_label(text):
    t = toks(text)
    return 'yes' if ('yes' in t and 'no' not in t) else ('no' if ('no' in t and 'yes' not in t) else 'other')

def recall(pred, gt):
    g, p = Counter(toks(gt)), Counter(toks(pred))
    return sum((g & p).values()) / sum(g.values()) if g else 0.0

val = json.load(open(a.val_json))
if a.kind == 'yn':
    rows = [r for r in val if r['answer_type'] == 'CLOSED' and r['answer'].strip().lower() in ('yes', 'no')]
    if a.n and a.n < len(rows):
        rows = random.Random(a.seed).sample(rows, a.n)
else:
    rows = [r for r in val if r['answer_type'] == 'OPEN']
    k = a.n or 300
    if k < len(rows):
        rows = random.Random(a.seed).sample(rows, k)
n = len(rows)
shift = max(1, n // 2)
print(f'[{a.kind}] val 문항 {n}개')

@torch.no_grad()
def run(image_of):
    out_texts = []
    for i in range(0, n, a.gen_bs):
        idx = list(range(i, min(n, i + a.gen_bs)))
        seqs = [prompt_ids(rows[k]['question']) for k in idx]
        L = max(len(s) for s in seqs)
        ids = torch.tensor([[tok.pad_token_id] * (L - len(s)) + s for s in seqs]).cuda()
        att = torch.tensor([[0] * (L - len(s)) + [1] * len(s) for s in seqs]).cuda()
        imgs = torch.stack([pixel(rows[image_of(k)]['image']) for k in idx]).to(device='cuda', dtype=dtype)
        out = model.generate(input_ids=ids, attention_mask=att, images=imgs, do_sample=False, max_new_tokens=a.gen_new, use_cache=True)
        out_texts += [re.sub(r'^Assistant:\s*', '', t.split('###')[0].strip()).strip() for t in tok.batch_decode(out[:, L:], skip_special_tokens=True)]
    return out_texts

res = {}
for name, f in (('맞는 이미지', lambda k: k), ('다른 문항의 이미지', lambda k: (k + shift) % n)):
    p = run(f)
    if a.kind == 'yn':
        lab = [yn_label(t) for t in p]
        sc = 100 * sum(l == r['answer'].strip().lower() for l, r in zip(lab, rows)) / n
        print(f'[{name}] yes/no 정확도 {sc:.1f}% | 예측 분포 {dict(Counter(lab))}', flush=True)
    else:
        sc = 100 * sum(recall(t, r['answer']) for t, r in zip(p, rows)) / n
        same = sum(toks(t) == toks(r['answer']) for t, r in zip(p, rows))
        print(f'[{name}] open recall {sc:.1f}% | 정답과 완전 일치 {same}/{n}', flush=True)
    res[name] = (sc, p)
d = res['맞는 이미지'][0] - res['다른 문항의 이미지'][0]
changed = sum(x != y for x, y in zip(res['맞는 이미지'][1], res['다른 문항의 이미지'][1]))
print(f'\n점수 차이(맞는 이미지 − 다른 이미지) = {d:+.1f}%p | 이미지를 바꾸자 답이 달라진 문항 {changed}/{n}')
print('해석: 차이가 크면 모델이 이미지를 쓰고 있다. 거의 0이고 달라진 문항도 적으면 이미지를 거의 보지 않고 질문 문구/사전 분포로 답한다.')


Writing /content/LLaVA-Med/ablate_val.py


FileNotFoundError: [Errno 2] No such file or directory: '/content/LLaVA-Med/ablate_val.py'

In [ ]:
# C-9a2. (진단) 이미지 사용 여부 점검 — 같은 질문에 맞는 이미지 vs 다른 문항의 이미지를 넣고 점수를 비교한다 (val 만 사용, 질문 종류별 약 5~10분).
# 앞의 %%writefile 셀을 먼저 실행해 스크립트를 (갱신해서) 저장해 둘 것.
assert 'SEL' in globals(), '먼저 C-9 로 학습률을 선택하세요'
KIND = 'open'                          # 'yn': yes/no 정확도 | 'open': open 문항 recall
ADAPTER = ''      # '' 로 바꾸면 어댑터 없이 출발점(instruct) 모델을 평가 → 학습이 이미지 활용을 바꿨는지 비교
adp = f'--adapter {ADAPTER}' if ADAPTER else ''
rc = tee(f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {QPY} ablate_val.py --model {INSTRUCT_MODEL} {adp} --kind {KIND} '
         f'--val_json {QDATA}/slake_val.json --image_folder {SLAKE_IMG_DIR}')
print('종료 코드', rc)

/content/venv_qlora/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.76s/it]
You are using the legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This means that tokens that come after special tokens will not be properly handled. We recommend you to read the related pull request available at https://github.com/huggingface/transformers/pull/24565
/content/venv_qlora/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
adapter: /content/drive/MyDrive/exp

In [27]:
# C-9c. (선택) 2단계: C-9 에서 val 로 선택된 학습률 하나만 FINAL_EPOCHS epoch 로 길게 학습 (백그라운드). 생략하면 C-9 의 선택(짧은 학습)을 그대로 쓴다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률을 선택하세요'
if 'final_proc' in globals() and final_proc.poll() is None:
    print('이미 실행 중입니다. PID', final_proc.pid)
else:
    SEARCH_SEL = SEL                      # 1단계 선택 결과 보관
    d = lr_dir(FINAL_STAGE, SEL['lr']); os.makedirs(d, exist_ok=True)
    cmd = train_cmd(SEL['lr'], FINAL_STAGE, f'--epochs {FINAL_EPOCHS}') + f' >> {d}/stdout.log 2>&1'
    final_proc = subprocess.Popen(['bash', '-c', cmd], start_new_session=True)
    print(f'lr={SEL["lr"]:g} 로 {FINAL_EPOCHS} epoch 학습 시작. PID', final_proc.pid, '| 진행은 C-8 과 같은 방식으로 아래 로그/val_metrics 를 확인:', d)

lr=0.0002 로 6 epoch 학습 시작. PID 56763 | 진행은 C-8 과 같은 방식으로 아래 로그/val_metrics 를 확인: /content/drive/MyDrive/expert_cfg_llavamed/qlora_slake/final_proj/lr0.0002


In [26]:
import os, signal; os.killpg(final_proc.pid, signal.SIGTERM)

In [45]:
# C-9d. 2단계 결과로 선택 갱신: 긴 학습의 val 결과에서 epoch 를 다시 고른다 (train 손실이 아니라 val 성능). 이후 C-9b/C-10 이 이 선택을 쓴다.
lr_ = SEARCH_SEL['lr'] if 'SEARCH_SEL' in globals() else SEL['lr']
d = lr_dir(FINAL_STAGE, lr_)
print('프로세스 종료 코드(None=실행 중):', final_proc.poll() if 'final_proc' in globals() else '(미실행)')
vm = f'{d}/val_metrics.jsonl'
if not os.path.exists(vm):
    lg = f'{d}/stdout.log'
    tail = ''.join(open(lg, errors='ignore').readlines()[-12:]) if os.path.exists(lg) else '(로그 파일 없음 → C-9c 를 아직 실행하지 않았거나 시작 전에 실패)'
    raise AssertionError('아직 val 결과가 없습니다. 첫 epoch 가 끝나야 생깁니다(약 30분). 학습 프로세스가 죽었는지 아래 로그 끝을 확인하세요.\n--- ' + lg + ' 끝 ---\n' + tail)
recs = [json.loads(l) for l in open(vm)]
for v in recs:
    print(f'  epoch {v["epoch"]:.2f}  train_loss {v["train_loss"]:.4f}  val_loss {v["eval_loss"]:.4f}  closed {v["closed"]:.2f}  open recall {v["open_recall"]:.2f}  overall {v["overall"]:.2f}')
assert os.path.exists(f'{d}/run_info.json'), '긴 학습이 아직 끝나지 않았습니다 (run_info.json 없음)'
info = json.load(open(f'{d}/run_info.json'))
b = best_of(recs)
if FINAL_TRAINVAL:      # train+val 학습: val 선택이 의미 없으므로 마지막 epoch 를 사용한다
    b = recs[-1]; info['best_checkpoint'] = info['last_checkpoint']
    print('※ train+val 학습(규칙 일부 완화): val 선택 없이 마지막 epoch 사용 — 결과표에 별도 표기하세요')
runs_search, runs = runs, {lr_: recs}
SEL = {'lr': lr_, 'epoch': b['epoch'], 'best_checkpoint': info['best_checkpoint'], 'last_checkpoint': info['last_checkpoint'], 'val': b, 'dir': d}
print(f"\n갱신된 선택: lr={lr_:g}, epoch {b['epoch']:.0f} / {recs[-1]['epoch']:.0f} (val overall {b['overall']:.2f}, closed {b['closed']:.2f}, open recall {b['open_recall']:.2f})")

In [46]:
# C-9b. test 문항 파일 준비 + 겹침 점검 — 여기서 처음 test 를 읽는다. C-9 에서 학습률·epoch 선택이 끝난 뒤에만 실행한다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률·epoch 를 선택하세요'
slake_test = [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', 'test.json', repo_type='dataset'))) if r['q_lang'] == 'en']
assert len(slake_test) == 1061, len(slake_test)
for r in slake_test:
    r['answer_type'] = r['answer_type'].upper()
write_eval_files('slake', slake_test, lambda r: r['img_name'])
assert all(os.path.exists(os.path.join(SLAKE_IMG_DIR, r['img_name'])) for r in slake_test), '이미지 누락'

# 분할 간 겹침 (이미지+질문): 학습 데이터와 test 가 겹치면 결과 해석에 주의
_trn, _val = json.load(open(f'{QDATA}/slake_train.json')), json.load(open(f'{QDATA}/slake_val.json'))
_k = lambda img, q: (img, q)
_te = {_k(r['img_name'], r['question'] + (CLOSED_SUFFIX if r['answer_type'] == 'CLOSED' else '')) for r in slake_test}
print('train∩test:', len({_k(r['image'], r['question']) for r in _trn} & _te), '| val∩test:', len({_k(r['image'], r['question']) for r in _val} & _te))

test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

slake: 1061문항 (OPEN 645 / CLOSED 416)
train∩test: 0 | val∩test: 0


In [47]:
# C-10. 선택된 설정에 대해서만 test 추론 (여기서 처음 test 를 읽는다). 어댑터 병합 → 기존 추론 경로로 평가. 병합 모델은 약 13GB 라 끝나면 지워도 된다.
assert 'SEL' in globals(), '먼저 C-9 로 학습률·epoch 를 선택하세요'
assert 'do_sample=False' in open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read(), 'greedy 패치(앞 절)가 필요합니다'
TEST_QFILE = f'{EVAL_DIR}/slake_test.json'
assert os.path.exists(TEST_QFILE), '먼저 C-9b 로 test 문항 파일을 만드세요'

def merge_and_infer(tag, ckpt):
    out = f'/content/llava-med-7b-slake-qlora-{tag}'
    sh(f'cd /content/LLaVA-Med && {QPY} merge_qlora.py --base {INSTRUCT_MODEL} --adapter {ckpt} --out {out}')
    cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py --model-name {out} '
           f'--question-file {TEST_QFILE} --image-folder {SLAKE_IMG_DIR} --answers-file {DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl '
           f'> /content/slake_qlora_{tag}.log 2>&1')
    return subprocess.Popen(['bash', '-c', cmd], start_new_session=True)

test_procs = {'best': merge_and_infer('best', SEL['best_checkpoint'])}
print('best(val 선택 epoch) 추론 PID', test_procs['best'].pid)

/content/venv_qlora/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.28s/it]
어댑터에 mm_projector 가중치가 없습니다 (LoRA 만 학습한 어댑터)
merged -> /content/llava-med-7b-slake-qlora-best
best(val 선택 epoch) 추론 PID 114598


In [61]:
# C-11. (선택) 마지막 epoch 체크포인트도 같은 방식으로 추론 — 보고표의 "Last-epoch overall" 용. best 가 마지막 epoch 와 같으면 건너뛴다.
if SEL['last_checkpoint'] == SEL['best_checkpoint']:
    print('best 가 마지막 epoch 와 같습니다 → 같은 결과를 재사용')
else:
    assert test_procs['best'].poll() is not None, '먼저 best 추론이 끝나야 합니다 (디스크/GPU 점유)'
    test_procs['last'] = merge_and_infer('last', SEL['last_checkpoint'])
    print('last 추론 PID', test_procs['last'].pid)

best 가 마지막 epoch 와 같습니다 → 같은 결과를 재사용


In [69]:
# C-12. 진행 확인
for tag, pr in test_procs.items():
    p = f'{DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl'
    n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
    print(f'{tag}: {n}/1061 | 종료 코드(None=실행 중): {pr.poll()}')

best: 1061/1061 | 종료 코드(None=실행 중): 0


In [70]:
# C-13. 최종 보고표 (기존 채점: closed accuracy / open recall). Best epoch 는 val 로 고른 값이다.
if FINAL_TRAINVAL:
    print('※ SLAKE QLoRA: train+val 학습 + 9 epoch 마지막 epoch (규칙 일부 완화). 규칙 준수 결과(train 만 학습, val 선택, 6 epoch: 79.40)와 별도 행으로 표기하세요.\n')
paper_slake = {'open': 83.08, 'closed': 85.34, 'overall': 83.97}
res = {}
for tag in ('best', 'last'):
    p = f'{DRIVE_ROOT}/slake_qlora_{tag}_pred.jsonl'
    if tag == 'last' and SEL['last_checkpoint'] == SEL['best_checkpoint']:
        res['last'] = res.get('best'); continue
    if os.path.exists(p) and sum(1 for _ in open(p)) == 1061:
        res[tag] = score_jsonl(f'{EVAL_DIR}/slake_ref.json', p)
m, ml = res.get('best'), res.get('last')
assert m is not None, 'best 추론이 아직 끝나지 않았습니다'
epochs = int(round(runs[SEL['lr']][-1]['epoch']))
print(f"{'Dataset':8s} {'LR':>8s} {'Epochs':>6s} {'Best epoch(val)':>15s} {'Open':>7s} {'Closed':>7s} {'Overall':>8s} {'Paper overall':>13s} {'Delta':>7s} {'Last-epoch overall':>19s}")
print(f"{'SLAKE':8s} {SEL['lr']:8g} {epochs:6d} {int(round(SEL['epoch'])):15d} {m['open']:7.2f} {m['closed']:7.2f} {m['overall']:8.2f} {paper_slake['overall']:13.2f} {m['overall'] - paper_slake['overall']:+7.2f} "
      + (f"{ml['overall']:19.2f}" if ml else f"{'(미실행)':>19s}"))
print('※ QLoRA 근사: 논문 LLaVA-Med(전체 파인튜닝)와 조건이 다르므로 Delta 는 재현 오차가 아니라 방식 차이를 포함합니다. 학습률·epoch 는 val 로만 선택, test 는 위 추론에서 처음 사용.')
json.dump({'selected': SEL, 'test_best': m, 'test_last': ml}, open(f'{QROOT}/final_report.json', 'w'), indent=1, default=str)

In [1]:
%%writefile /content/curve_plot.py
"""모델 공통 학습 곡선 그림 (2x2). 선택 기준은 항상 val 이고, test 곡선은 진단용으로만 겹쳐 그린다.

  (a) train loss (선택: 학습률)
  (b) val overall vs test overall, 세로선 = val 로 고른 epoch
  (c) epoch 별 open / closed / overall  (실선 = val, 점선 = test, 회색 가로 점선 = 논문 test 값)
  (d) "이 epoch 에서 학습을 멈췄다면" 보고했을 값: 그 시점까지 val overall 이 가장 높았던 epoch 의 test 값 (test 가 있는 epoch 만).
      test 가 없으면 val 의 best-so-far 를 그린다. 어느 쪽이든 선택에 test 는 쓰지 않는다.

입력 형식 (epoch 는 0 부터 센 정수 또는 1 부터 센 정수 중 하나로 통일해서 넣을 것)
  train_loss : {epoch: float}
  lr         : {epoch: float}                (선택)
  val, test  : {epoch: {'open': %, 'closed': %, 'overall': %}}   test 는 평가한 epoch 만 있어도 된다
  paper      : {'open': %, 'closed': %, 'overall': %}            (선택)
"""
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

KEYS = ('open', 'closed', 'overall')
COL = {'open': 'tab:blue', 'closed': 'tab:orange', 'overall': 'tab:green'}


def select_by_val(val):
    """val overall 이 최대인 epoch (동점이면 이른 epoch)."""
    return max(sorted(val), key=lambda e: (val[e]['overall'], -e))


def best_so_far(val):
    """각 epoch 까지의 val-best epoch 를 {epoch: 그때의 best epoch} 로 돌려준다."""
    out, best = {}, None
    for e in sorted(val):
        if best is None or val[e]['overall'] > val[best]['overall']:
            best = e
        out[e] = best
    return out


def _sparse(test, val):
    """test 가 5개 미만(보통 val 선택 epoch 와 마지막 epoch 두 점)이면 곡선이 아니므로 report 레이아웃(막대 비교)을 쓴다.
    5 epoch 마다 평가하는 등 5개 이상이면 점을 선으로 이은 곡선 레이아웃을 쓴다."""
    return len(test) < 5


def _bars(a, groups, series, ylim=None):
    """groups: 막대 묶음 이름 목록, series: [(label, [값...], 스타일 dict)]"""
    n = len(series); w = 0.8 / n
    for j, (lab, vals, kw) in enumerate(series):
        xs = [i + (j - (n - 1) / 2) * w for i in range(len(groups))]
        a.bar(xs, [0 if v is None else v for v in vals], w * 0.95, label=lab, **kw)
        for x, v in zip(xs, vals):
            if v is not None:
                a.text(x, v + 0.8, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
    a.set_xticks(range(len(groups))); a.set_xticklabels(groups); a.set_ylabel('%')
    if ylim:
        a.set_ylim(*ylim)


def _ticks(axes, val, test, train_loss):
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in axes:
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d')); a.grid(axis='x', alpha=0.2)


def _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, sel):
    """test 를 한두 번만 평가한 경우의 레이아웃: 선택 근거(val) + 최종 test 비교 + 후보 coverage."""
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))
    # (a) train loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss); a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss'); a.set_yscale('log'); a.legend(loc='upper right')
    else:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss'); a.set_xlabel('epoch')
    # (b) val 곡선 = 선택 근거
    a = ax[0, 1]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.6 if k == 'overall' else 1.0, label=f'val {k}')
    a.axvline(sel, color='green', lw=1.0, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation per epoch (the only signal used for selection)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    # (c) test 비교
    a = ax[1, 0]
    series = []
    if sel in test:
        series.append((f'val-selected (epoch {sel})', [test[sel][k] for k in KEYS], dict(color='tab:blue')))
    last = max(test) if test else None
    if last is not None and last != sel:
        series.append((f'last epoch ({last})', [test[last][k] for k in KEYS], dict(color='tab:cyan')))
    if paper:
        series.append(('paper', [paper.get(k) for k in KEYS], dict(color='lightgray', hatch='//', edgecolor='gray')))
    if series:
        _bars(a, KEYS, series, ylim=(0, 120)); a.legend(loc='upper center', fontsize=8, ncol=3)
    a.set_title('(c) test score, evaluated once per checkpoint (never used for selection)', fontsize=10)
    # (d) candidate coverage
    a = ax[1, 1]
    if coverage:
        cov = [coverage[k]['coverage'] if isinstance(coverage[k], dict) else coverage[k] for k in KEYS]
        _bars(a, KEYS, [('candidate coverage (answer is in the train-only list)', cov, dict(color='lightsteelblue'))], ylim=(0, 120))
        if sel in test:
            a.plot(range(len(KEYS)), [test[sel][k] for k in KEYS], ls='none', marker='D', ms=8, color='tab:blue', markeredgecolor='k', label=f'test score (epoch {sel})')
        a.set_title('(d) candidate coverage of test answers vs achieved score', fontsize=10); a.legend(loc='upper center', fontsize=8, ncol=2)
    else:
        bs = best_so_far(val); xs = sorted(bs)
        for k in KEYS:
            a.step(xs, [val[bs[e]][k] for e in xs], where='post', color=COL[k], label=f'val {k}')
        a.axvline(sel, color='green', lw=1.0)
        a.set_title('(d) val best-so-far (selection trace)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    _ticks([ax[0, 0], ax[0, 1]] + ([] if coverage else [ax[1, 1]]), val, test, train_loss)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


def closed_confusion(y_true, y_pred):
    """yes/no 문항의 혼동행렬과 클래스별 precision / recall / F1. y_pred 가 yes/no 가 아니면 'other' 로 센다.
    입력은 소문자·구두점 정리된 문자열 목록. 점수(확률)가 아니라 예측 답만 쓴다."""
    cls = ('yes', 'no'); cols = ('yes', 'no', 'other')
    cm = {t: {c: 0 for c in cols} for t in cls}
    for t, p in zip(y_true, y_pred):
        if t in cm:
            cm[t][p if p in ('yes', 'no') else 'other'] += 1
    m = {}
    for c in cls:
        tp = cm[c][c]; pred = sum(cm[t][c] for t in cls); true = sum(cm[c].values())
        pr = tp / pred if pred else 0.0; rc = tp / true if true else 0.0
        m[c] = {'precision': 100 * pr, 'recall': 100 * rc, 'f1': 100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 'support': true}
    m['macro'] = {k: sum(m[c][k] for c in cls) / 2 for k in ('precision', 'recall', 'f1')}
    n = sum(sum(r.values()) for r in cm.values())
    m['accuracy'] = 100 * sum(cm[c][c] for c in cls) / n if n else 0.0
    return cm, m


def plot_closed_f1(title, y_true, y_pred, out_png=None):
    """closed(yes/no) 예측의 혼동행렬 + 클래스별 precision/recall/F1 막대. (ROC 는 확률 점수가 있어야 해서 여기서는 그리지 않는다)"""
    cm, m = closed_confusion(y_true, y_pred)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    cols = ('yes', 'no', 'other'); rows = ('yes', 'no')
    a = ax[0]; a.imshow([[cm[r][c] for c in cols] for r in rows], cmap='Blues')
    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            a.text(j, i, cm[r][c], ha='center', va='center', fontsize=14, color='black')
    a.set_xticks(range(3)); a.set_xticklabels([f'pred {c}' for c in cols]); a.set_yticks(range(2)); a.set_yticklabels([f'true {r}' for r in rows])
    a.set_title(f'confusion matrix (closed, n={sum(sum(v.values()) for v in cm.values())})')
    groups = ('yes', 'no', 'macro')
    _bars(ax[1], groups, [(k, [m[g][k] for g in groups], dict(color=c)) for k, c in (('precision', 'tab:blue'), ('recall', 'tab:orange'), ('f1', 'tab:green'))], ylim=(0, 120))
    ax[1].legend(loc='upper center', fontsize=8, ncol=3); ax[1].set_title(f"precision / recall / F1  (accuracy {m['accuracy']:.1f}%)")
    fig.suptitle(title, fontsize=11); fig.tight_layout(rect=(0, 0, 1, 0.94))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, m


def plot_curves(title, train_loss=None, lr=None, val=None, test=None, paper=None, out_png=None, selected=None, coverage=None):
    val, test = val or {}, test or {}
    sparse = _sparse(test, val)
    if sparse:      # test 가 한두 번만 평가된 경우: 우리 방식(val 선택, test 한 번)에 맞는 레이아웃
        assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
        return _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, select_by_val(val) if selected is None else selected)
    tkw = dict(ls='none', marker='o', ms=7) if sparse else dict(marker='o', ms=4)
    assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
    sel = select_by_val(val) if selected is None else selected
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))

    # (a) loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss)
        a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss')
        a.set_yscale('log')
    if not train_loss:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss' + (' / learning rate' if lr else '')); a.set_xlabel('epoch')
    h, l = a.get_legend_handles_labels()
    if lr:
        b = a.twinx(); xs = sorted(lr)
        b.plot(xs, [lr[e] for e in xs], ':', color='gray', label='lr'); b.set_ylabel('lr')
        h2, l2 = b.get_legend_handles_labels(); h += h2; l += l2
    if h:
        a.legend(h, l, loc='upper right')

    # (b) val vs test overall
    a = ax[0, 1]
    xs = sorted(val); a.plot(xs, [val[e]['overall'] for e in xs], marker='.', color='tab:orange', label='validation overall')
    if test:
        xt = sorted(test); a.plot(xt, [test[e]['overall'] for e in xt], color='tab:blue', label='test overall (diagnostic)', **tkw)
    a.axvline(sel, color='green', lw=0.8, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation vs test overall'); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    # (c) per-epoch metrics
    a = ax[1, 0]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.2, label=f'val {k}')
        if test:
            xt = sorted(test); a.plot(xt, [test[e][k] for e in xt], color=COL[k], label=f'test {k}', **({'ls': 'none', 'marker': 'o', 'ms': 7} if sparse else {'ls': '--', 'marker': 'o', 'ms': 3, 'lw': 0.9}))
        if paper and paper.get(k) is not None:
            a.axhline(paper[k], color=COL[k], ls=':', lw=0.9, alpha=0.7)
    a.set_title('(c) accuracy per epoch (solid = val, dashed = test, dotted = paper test)'); a.set_xlabel('epoch'); a.set_ylabel('%')
    a.legend(loc='lower right', fontsize=7, ncol=2)

    # (d) reported value if training stopped at this epoch (val-based selection)
    a = ax[1, 1]; bs = best_so_far(val)
    dense = len(test) >= max(5, 0.5 * len(val))      # test 가 epoch 의 절반 이상이어야 '그 시점 val-best 의 test' 계단을 그릴 수 있다
    if test and dense and any(bs[e] in test for e in bs):
        lab = 'test @ val-best-so-far epoch'
        for k in KEYS:
            pts = [(e, test[bs[e]][k]) for e in sorted(bs) if bs[e] in test]
            a.step([p[0] for p in pts], [p[1] for p in pts], where='post', color=COL[k], label=f'{k} (test)')
    else:
        lab = 'lines: val best-so-far | markers: test at evaluated epochs' if test else 'val best-so-far (no per-epoch test)'
        for k in KEYS:
            a.step(sorted(bs), [val[bs[e]][k] for e in sorted(bs)], where='post', color=COL[k], label=f'{k} (val)')
            if test:
                a.plot(sorted(test), [test[e][k] for e in sorted(test)], ls='none', marker='*', ms=11, color=COL[k], markeredgecolor='k', label=f'{k} (test)')
    if paper:
        for k in KEYS:
            if paper.get(k) is not None:
                a.axhline(paper[k], color='gray', ls=':', lw=0.7)
    a.axvline(sel, color='red', lw=0.8)
    a.set_title('(d) reported value if training stopped at this epoch\n[' + lab + ']', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    # x 축 눈금: epoch 가 15 이상이면 5 단위, 그보다 짧으면 1 단위
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in ax.ravel():
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d'))
        a.grid(axis='x', alpha=0.2)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


In [ ]:
# C-14. 학습 곡선 (선택 = val). test 는 val 로 고른 epoch(best)와 마지막 epoch(last)만 평가했으므로 두 점만 표시한다.
import sys
sys.path.insert(0, '/content')
import importlib, curve_plot; importlib.reload(curve_plot)      # 저장 셀로 갱신한 최신 코드를 쓰도록 다시 불러온다
from curve_plot import plot_curves
_recs = [json.loads(l) for l in open(f"{SEL['dir']}/val_metrics.jsonl")]
_ep = lambda r: int(round(r['epoch']))
val  = {_ep(r): {'open': r['open_recall'], 'closed': r['closed'], 'overall': r['overall']} for r in _recs}
loss = {_ep(r): r['train_loss'] for r in _recs if r.get('train_loss') is not None}
sel = int(round(SEL['epoch']))
test = {}
if res.get('best'): test[sel] = {k: res['best'][k] for k in ('open', 'closed', 'overall')}
if res.get('last'): test[max(val)] = {k: res['last'][k] for k in ('open', 'closed', 'overall')}
fig, _ = plot_curves(f"LLaVA-Med v1.0 + QLoRA (SLAKE, lr {SEL['lr']:g}, LoRA+mm_projector)", train_loss=loss, val=val, test=test,
                     paper=paper_slake, out_png=f'{QROOT}/curves.png', selected=sel)
print('저장:', f'{QROOT}/curves.png')


In [ ]:
# C-15. closed(yes/no) 혼동행렬 + precision / recall / F1 — val 로 고른 체크포인트(best)의 test 예측 (ROC 는 확률 점수가 없어 그리지 않는다)
import importlib, curve_plot; importlib.reload(curve_plot)
from curve_plot import plot_closed_f1
_refs = {int(k): v for k, v in json.load(open(f'{EVAL_DIR}/slake_ref.json')).items()}
_preds = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/slake_qlora_best_pred.jsonl')}
assert set(_preds) == set(_refs), '예측과 참조의 id 가 다릅니다 (추론이 끝났는지 확인)'
def _cls(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'other'
_ids = [i for i, r in _refs.items() if r['answer_type'] == 'CLOSED' and r['answer'].strip().lower() in ('yes', 'no')]
_n_closed = sum(r['answer_type'] == 'CLOSED' for r in _refs.values())
fig, m = plot_closed_f1(f"LLaVA-Med + QLoRA SLAKE closed yes/no (n={len(_ids)} of {_n_closed} closed)",
                        [_refs[i]['answer'].strip().lower() for i in _ids], [_cls(_preds[i]) for i in _ids], out_png=f'{QROOT}/closed_f1.png')
print(f"accuracy {m['accuracy']:.2f} | macro F1 {m['macro']['f1']:.2f} | yes F1 {m['yes']['f1']:.2f} (n={m['yes']['support']}) | no F1 {m['no']['f1']:.2f} (n={m['no']['support']})")
print(f'※ closed 문항 중 정답이 yes/no 가 아닌 {_n_closed - len(_ids)}개는 F1 에서 제외했다 (원 채점에서는 이런 문항이 항상 0점)')


In [3]:
# C-16. VQA-RAD(공식 delta) closed 예측 진단 — 이 셀만 단독으로 실행해도 된다 (다른 셀 변수에 의존하지 않음).
# 예측이 yes / no / 그 외 중 무엇인지 본다 (프롬프트·디코딩 변형을 고르기 전에 원인 확인).
import json, os, re, glob
from collections import Counter
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
def normalize(text):
    text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
    return re.sub(r'[^a-z0-9\s]', ' ', text).split()
_REF, _PRED = f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'
if not os.path.exists(_REF):      # 런타임 초기화로 /content 가 비었을 때: 데이터 셀(C-1)과 같은 방식으로 정답 파일을 다시 만든다
    _src = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
    assert os.path.exists(_src), f'{_src} 가 없습니다'
    _all = json.load(open(_src))
    _test = [r for r in _all if str(r.get('phrase_type', '')).lower() in ('test_freeform', 'test_para')]
    assert len(_test) == 451, len(_test)
    _at = lambda a: 'CLOSED' if str(a).strip().lower() in ('yes', 'no') else 'OPEN'
    os.makedirs(EVAL_DIR, exist_ok=True)
    json.dump({i: {'answer': str(r['answer']), 'answer_type': str(r.get('answer_type', _at(r.get('answer', '')))).upper()}
               for i, r in enumerate(_test)}, open(_REF, 'w'))
    print('정답 파일 재생성:', _REF)
assert os.path.exists(_PRED), f'{_PRED} 가 없습니다 (VQA-RAD greedy 추론 결과)'
refs = {int(k): v for k, v in json.load(open(_REF)).items()}
preds = {}
for _l in open(_PRED):
    _p = json.loads(_l); preds[int(_p['question_id'])] = _p['text']
def cls(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'both' if y and n else 'other'
rows = [(i, refs[i]['answer'].strip().lower(), cls(preds[i]), preds[i]) for i in refs if refs[i]['answer_type'] == 'CLOSED']
yn = [r for r in rows if r[1] in ('yes', 'no')]
print(f'closed {len(rows)}문항 (정답이 yes/no 인 것 {len(yn)}, 아닌 것 {len(rows) - len(yn)})')
print('정답 분포 :', dict(Counter(r[1] if r[1] in ('yes', 'no') else 'other' for r in rows)))
print('예측 분류 :', dict(Counter(r[2] for r in rows)))
print(f'정답이 yes/no 인 {len(yn)}문항 중: 맞음 {sum(r[1] == r[2] for r in yn)} | yes/no 로 답했으나 틀림 {sum(r[2] in ("yes", "no") and r[1] != r[2] for r in yn)} | yes/no 가 아닌 답 {sum(r[2] not in ("yes", "no") for r in yn)}')
print('혼동 (정답→예측):', dict(Counter((r[1], r[2]) for r in yn)))
print('\n예측이 yes/no 로 분류되지 않은 예시 (최대 10):')
for r in [r for r in rows if r[2] in ('other', 'both')][:10]:
    print('  정답', r[1][:20].ljust(20), '←', r[3][:100].replace('\n', ' '))


정답 파일 재생성: /content/llava_eval_data/vqarad_ref.json
closed 272문항 (정답이 yes/no 인 것 251, 아닌 것 21)
정답 분포 : {'yes': 118, 'no': 133, 'other': 21}
예측 분류 : {'no': 101, 'yes': 165, 'other': 6}
정답이 yes/no 인 251문항 중: 맞음 192 | yes/no 로 답했으나 틀림 59 | yes/no 가 아닌 답 0
혼동 (정답→예측): {('yes', 'no'): 9, ('yes', 'yes'): 109, ('no', 'no'): 83, ('no', 'yes'): 50}

예측이 yes/no 로 분류되지 않은 예시 (최대 10):
  정답 left                 ← Assistant: Left side
  정답 supratentorial       ← Assistant: supratentorial
  정답 left                 ← Assistant: Right
  정답 right side           ← Assistant: right side
  정답 both                 ← Assistant: both sides
  정답 flair                ← Assistant: FLAIR


In [31]:
# C-17. VQA-RAD closed 프롬프트 변형 (test 기반 탐색 — 규칙 완화, 별도 행으로 표기할 것).
# 접미어 "[yes, no]" 를 정답 유형이 아니라 '질문 문장'으로 판단해 붙인다: yes/no 형태 질문에만 붙이고, 선택지가 있는 질문("left or right?" 등)은 접미어 없이 원문 그대로 묻는다.
# 이미 있는 greedy 예측(vqarad_pred_greedy.jsonl)을 재사용하고, 접미어를 빼는 문항만 다시 추론하므로 빠르다. 모델 로딩 때문에 1~2분 정도 걸린다(추정).
import json, re, os
from collections import Counter
import numpy as np
if 'score_jsonl' not in globals():      # 채점 함수 셀을 실행하지 않았어도 이 셀만으로 채점되도록 같은 정의를 둔다 (본문 채점 셀과 동일한 방식)
    def normalize(text):
        text = str(text).lower().replace('<|end|>', ' ').replace('###', ' ')
        return re.sub(r'[^a-z0-9\s]', ' ', text).split()
    def closed_accuracy(pred, gt):
        return float(gt.strip().lower() in normalize(pred))
    def open_recall(pred, gt):
        g, p = Counter(normalize(gt)), Counter(normalize(pred))
        return sum((g & p).values()) / sum(g.values()) if g else 0.0
    def score_jsonl(ref_file, pred_file):
        refs = {int(k): v for k, v in json.load(open(ref_file)).items()}
        preds = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(pred_file)}
        assert set(preds) == set(refs), f'예측 {len(preds)}개 / 참조 {len(refs)}개'
        cl, op = [], []
        for i, r in refs.items():
            (cl if r['answer_type'] == 'CLOSED' else op).append(closed_accuracy(preds[i], r['answer']) if r['answer_type'] == 'CLOSED' else open_recall(preds[i], r['answer']))
        return {'open': 100 * np.mean(op), 'closed': 100 * np.mean(cl), 'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}
SUF = ' Please choose from the following two options: [yes, no]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)

qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json'))
ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
redo = []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF): v = v[:-len(SUF)]
    if not is_yesno_form(v):
        q = json.loads(json.dumps(q)); q['conversations'][0]['value'] = v; redo.append(q)
json.dump(redo, open(f'{EVAL_DIR}/vqarad_closed_qform.json', 'w'))
# 분류가 맞았는지 확인용 (정답은 채점·점검에만 사용, 프롬프트 결정에는 쓰지 않음)
_gt = Counter('yes/no' if ref[str(q['id'])]['answer'].strip().lower() in ('yes', 'no') else 'other' for q in redo)
print(f'접미어 없이 다시 묻는 closed 문항: {len(redo)}개 (정답이 yes/no 인 것 {_gt["yes/no"]}, 아닌 것 {_gt["other"]})')
for q in redo[:8]: print('  ', q['conversations'][0]['value'][:90])

# 추론 스크립트가 greedy 가 아니면(do_sample=True, temperature 0.7) 기준 예측(greedy)과 비교가 어긋나므로 먼저 greedy 로 패치한다
_sp = '/content/LLaVA-Med/llava/eval/model_vqa_med.py'
_s = open(_sp).read()
_pat = r"do_sample=True,\s*temperature=0\.7,\s*max_new_tokens=1024,"
if re.search(_pat, _s):
    open(_sp, 'w').write(re.sub(_pat, "do_sample=False,\n                max_new_tokens=1024,", _s, count=1))
    print('추론 스크립트를 greedy 로 패치했습니다')
else:
    print('추론 스크립트: 이미 greedy')
%cd /content/LLaVA-Med
_out = f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl'
if os.path.exists(_out): os.remove(_out)
!PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_med.py \
    --model-name /content/llava-med-7b-vqarad \
    --question-file {EVAL_DIR}/vqarad_closed_qform.json \
    --image-folder "{VQARAD_IMG_DIR}" \
    --answers-file {_out} > /content/vqarad_qform.log 2>&1

new = [json.loads(l) for l in open(_out)]
assert len(new) == len(redo), f'예측 {len(new)}/{len(redo)} — /content/vqarad_qform.log 를 확인하세요'
old = [json.loads(l) for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')]
ids = {p['question_id'] for p in new}
with open(f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl', 'w') as f:
    for p in [p for p in old if p['question_id'] not in ids] + new: f.write(json.dumps(p) + '\n')
print('\n기준(전 문항 접미어, greedy):', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'))
print('변형(질문 형태 기준 접미어)  :', score_jsonl(f'{EVAL_DIR}/vqarad_ref.json', f'{DRIVE_ROOT}/vqarad_pred_qform.jsonl'))
print('※ test 기반 탐색 결과입니다. 규칙 준수 행과 별도 행으로 표기하세요.')


In [32]:
# C-18. C-17 에서 접미어를 뺀 문항을 하나씩 본다 — 어떤 질문이었고, 모델이 뭐라고 답했고, 접미어 때문에 바뀐 것은 무엇인지. (단독 실행 가능, 추론 없음)
import json, re, os
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
norm = lambda s: re.sub(r'[^a-z0-9\s]', ' ', str(s).lower().replace('<|end|>', ' ').replace('###', ' ')).split()
ref  = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
redo = json.load(open(f'{EVAL_DIR}/vqarad_closed_qform.json'))
ld   = lambda f: {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f)}
base, new = ld(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl'), ld(f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl')
ok = lambda pred, gt: gt.strip().lower() in norm(pred)
rows = [(q['id'], q['conversations'][0]['value'], ref[str(q['id'])]['answer']) for q in redo]
print(f'접미어를 뺀 문항 {len(rows)}개 | 기준 정답 {sum(ok(base[i], g) for i, _, g in rows)} → 변형 정답 {sum(ok(new[i], g) for i, _, g in rows)}\n')
for i, qtext, g in rows:
    b, n = ok(base[i], g), ok(new[i], g)
    mark = '↑개선' if (n and not b) else '↓악화' if (b and not n) else ('유지(정답)' if n else '유지(오답)')
    print(f'[{mark}] 정답={g[:25]!r:28} Q: {qtext[:70]}\n         기준→{base[i][:50].replace(chr(10), " ")!r}  |  변형→{new[i][:60].replace(chr(10), " ")!r}')


접미어를 뺀 문항 24개 | 기준 정답 6 → 변형 정답 11

[↑개선] 정답='one'                        Q: Are there multiple or just 1 metastatic focus?
         기준→'Assistant: no'  |  변형→'Assistant: Just one'
[유지(정답)] 정답='Left'                       Q: Is the colon more prominent on the patient's right or left side?
         기준→'Assistant: Left side'  |  변형→'Assistant: Left side'
[유지(오답)] 정답='Smaller'                    Q: Is the heart size in this image smaller or larger than if the image wa
         기준→'Assistant: yes'  |  변형→'Assistant: AP'
[유지(정답)] 정답='supratentorial'             Q: Is this supratentorial or infratentorial?
         기준→'Assistant: supratentorial'  |  변형→'Assistant: Supratentorial'
[유지(오답)] 정답='MRI'                        Q: Is this an MRI or a CT scan?
         기준→'Assistant: no'  |  변형→'Assistant: CT'
[유지(오답)] 정답='Left'                       Q: Is the lesion on the left or right side of the brain?
         기준→'Assistant: Right'  |  변형→'Assistant: Right'
[↑개선] 정답='Female'                     

In [36]:
# C-19. VQA-RAD yes/no 문항의 편향 보정 (test 기반 탐색 — 규칙 완화, 별도 행). 추론 + 분석을 한 셀에서 한다. C-17 을 먼저 (greedy 로) 실행해 둘 것.
# 모델이 "Assistant:" 다음에 내는 첫 답 토큰의 yes/no 로그확률 차이 d = log P(yes) - log P(no) 를 구한다.
# 선택지 순서를 [yes, no] / [no, yes] 로 바꾼 두 번을 물어 평균한다 (순서 편향 완화). 임계값 규칙 3가지를 비교한다:
#   0      : d > 0 이면 yes (보정 없음, 모델 자체 선호)
#   prior  : 예측 yes 비율이 train 의 yes 비율과 같아지도록 임계값을 정한다 (test 정답을 쓰지 않음)
#   oracle : test 정답으로 고른 최적 임계값 (상한 참고용 — 보고하지 말 것)
import json, re, os, subprocess
import numpy as np
from collections import Counter
DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/expert_cfg_llavamed')
EVAL_DIR   = globals().get('EVAL_DIR', '/content/llava_eval_data')
PY         = globals().get('PY', '/content/venv_llava/bin/python')
IMG_DIR    = globals().get('VQARAD_IMG_DIR', '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder')
PUBLIC     = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
SUF_A = ' Please choose from the following two options: [yes, no]'
SUF_B = ' Please choose from the following two options: [no, yes]'
AUX = ('is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'could', 'has', 'have', 'had', 'will', 'would', 'should')
def is_yesno_form(q):
    q = q.strip().lower()
    return q.split()[0].strip(',') in AUX and not re.search(r'\bor\b', q)
norm = lambda s: re.sub(r'[^a-z0-9\s]', ' ', str(s).lower().replace('<|end|>', ' ').replace('###', ' ')).split()

# 1) 질문 파일: closed 중 yes/no 형태 문항, 두 가지 선택지 순서 (순서 B 는 id + 100000)
qs  = json.load(open(f'{EVAL_DIR}/vqarad_test.json')); ref = json.load(open(f'{EVAL_DIR}/vqarad_ref.json'))
ynq, base_ids = [], []
for q in qs:
    if ref[str(q['id'])]['answer_type'] != 'CLOSED': continue
    v = q['conversations'][0]['value']
    if v.endswith(SUF_A): v = v[:-len(SUF_A)]
    if not is_yesno_form(v): continue
    base_ids.append(q['id'])
    for k, suf in enumerate((SUF_A, SUF_B)):
        qq = json.loads(json.dumps(q)); qq['id'] = q['id'] + 100000 * k; qq['conversations'][0]['value'] = v + suf; ynq.append(qq)
json.dump(ynq, open(f'{EVAL_DIR}/vqarad_yn2.json', 'w'))
print(f'yes/no 형태 closed 문항 {len(base_ids)}개 x 2가지 순서 = {len(ynq)}회 추론')

# 2) 추론 스크립트 복사본: greedy + 첫 답 토큰의 yes/no 로그확률을 .yn 파일에 기록
_src = open('/content/LLaVA-Med/llava/eval/model_vqa_med.py').read()
_pat = re.compile(r"with torch\.inference_mode\(\):\s*output_ids = model\.generate\(.*?stopping_criteria=\[stopping_criteria\]\)\n", re.S)
_n = len(_pat.findall(_src))
if _n == 0:      # 형태가 다르면 generate 주변을 그대로 보여 준다
    _k = _src.find('model.generate')
    print(_src[max(0, _k - 200):_k + 500] if _k >= 0 else 'model.generate 가 없습니다')
    print('CRLF 줄바꿈:', chr(13) in _src)
assert _n >= 1, 'model_vqa_med.py 의 generate 블록을 찾지 못했습니다 (위 출력을 알려 주세요)'
print(f'generate 블록 {_n}개 발견 -> 첫 번째만 교체')
_new = '''with torch.inference_mode():
            _go = model.generate(input_ids, images=images, do_sample=False, max_new_tokens=24,
                                 stopping_criteria=[stopping_criteria], output_scores=True, return_dict_in_generate=True)
        output_ids = _go.sequences
        _gen = output_ids[0, input_ids.shape[1]:]
        _t = 0
        for _k in range(len(_gen)):      # 'Assistant:' 이후의 첫 토큰 위치를 찾는다
            if ':' in tokenizer.decode(_gen[:_k + 1], skip_special_tokens=True):
                _t = _k + 1
                break
        _vy = [tokenizer.convert_tokens_to_ids(w) for w in ('yes', 'Yes', chr(9601) + 'yes', chr(9601) + 'Yes')]
        _vn = [tokenizer.convert_tokens_to_ids(w) for w in ('no', 'No', chr(9601) + 'no', chr(9601) + 'No')]
        _vy = [x for x in _vy if x is not None and x != tokenizer.unk_token_id]
        _vn = [x for x in _vn if x is not None and x != tokenizer.unk_token_id]
        _ly = _ln = None
        if _t < len(_go.scores):
            _lp = torch.log_softmax(_go.scores[_t][0].float(), -1)
            _ly = torch.logsumexp(_lp[_vy], 0).item()
            _ln = torch.logsumexp(_lp[_vn], 0).item()
        open(args.answers_file + '.yn', 'a').write(json.dumps({'question_id': idx, 'lp_yes': _ly, 'lp_no': _ln, 't': _t,
                                                              'head': tokenizer.decode(_gen[:_t + 3], skip_special_tokens=True)}) + chr(10))
'''
_YN = '/content/LLaVA-Med/llava/eval/model_vqa_yn.py'
open(_YN, 'w').write(_pat.sub(lambda m: _new, _src, count=1))

_out = f'{DRIVE_ROOT}/vqarad_pred_yn2.jsonl'
for f in (_out, _out + '.yn'):
    if os.path.exists(f): os.remove(f)
%cd /content/LLaVA-Med
import time
_cmd = (f'cd /content/LLaVA-Med && PYTHONUNBUFFERED=1 {PY} llava/eval/model_vqa_yn.py --model-name /content/llava-med-7b-vqarad '
        f'--question-file {EVAL_DIR}/vqarad_yn2.json --image-folder "{IMG_DIR}" --answers-file {_out} > /content/vqarad_yn2.log 2>&1')
_proc = subprocess.Popen(['bash', '-c', _cmd])
_t0 = time.time()
while _proc.poll() is None:      # 진행 상황: 로그확률이 기록된 문항 수 (1분마다 출력)
    _n = sum(1 for _ in open(_out + '.yn')) if os.path.exists(_out + '.yn') else 0
    print(f'[{(time.time() - _t0) / 60:5.1f}분] 진행 {_n}/{len(ynq)}', flush=True)
    time.sleep(60)
assert _proc.returncode == 0, f'추론 실패 (종료 코드 {_proc.returncode}) — /content/vqarad_yn2.log 의 끝부분을 확인하세요'
print(f'추론 완료 ({(time.time() - _t0) / 60:.1f}분)')

# 3) 분석
yn = {}
for l in open(_out + '.yn'):
    r = json.loads(l); yn[r['question_id']] = r
bad = [q['id'] for q in ynq if q['id'] not in yn or yn[q['id']]['lp_yes'] is None]
assert not bad, f'로그확률이 없는 문항 {len(bad)}개 — /content/vqarad_yn2.log 확인. 예: {bad[:5]}'
print('첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):')
for q in ynq[:4]: print('  ', yn[q['id']]['t'], repr(yn[q['id']]['head']))
dA = np.array([yn[i]['lp_yes'] - yn[i]['lp_no'] for i in base_ids])
dB = np.array([yn[i + 100000]['lp_yes'] - yn[i + 100000]['lp_no'] for i in base_ids])
gt = np.array([ref[str(i)]['answer'].strip().lower() for i in base_ids])
isyn = np.isin(gt, ['yes', 'no']); gy = (gt == 'yes')

pub = json.load(open(PUBLIC))
_tr = [r for r in pub if str(r.get('phrase_type', '')).lower() not in ('test_freeform', 'test_para') and str(r.get('answer', '')).strip().lower() in ('yes', 'no')]
p_yes = np.mean([str(r['answer']).strip().lower() == 'yes' for r in _tr])
print(f'train 의 yes 비율 {p_yes:.3f} (yes/no 정답 {len(_tr)}개) | 테스트 yes/no 문항 중 정답 yes 비율 {gy[isyn].mean():.3f}')

def acc(pred_yes): return int(((pred_yes == gy) & isyn).sum())          # 정답이 yes/no 가 아닌 문항은 항상 오답
srcs = {'순서A': dA, '순서B': dB, '평균': (dA + dB) / 2}
rules = {}
for name, d in srcs.items():
    cand = np.sort(d)
    best = max(acc(d > (a + b) / 2) for a, b in zip(cand[:-1], cand[1:]))
    rules[name] = {'0': acc(d > 0), 'prior': acc(d > np.quantile(d, 1 - p_yes)), 'oracle(상한)': best}

# 나머지 closed(선택형)는 C-17 의 변형 예측, 기준은 기존 greedy 예측을 쓴다
qf = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_closed_qform.jsonl')}
gr = {int(json.loads(l)['question_id']): json.loads(l)['text'] for l in open(f'{DRIVE_ROOT}/vqarad_pred_greedy.jsonl')}
n_closed = sum(v['answer_type'] == 'CLOSED' for v in ref.values())
other_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(t) for i, t in qf.items())
base_ok = sum(ref[str(i)]['answer'].strip().lower() in norm(gr[i]) for i in base_ids)
print(f'\nyes/no 형태 {len(base_ids)}문항 중 기준(접미어 [yes, no], greedy) 정답: {base_ok}개 (정답이 yes/no 인 문항 {int(isyn.sum())}개)')
print(f'{"입력":6}{"규칙":14}{"yes/no 정답수":>14}{"closed(272)":>14}')
for name, rr in rules.items():
    for rule, n in rr.items():
        print(f'{name:6}{rule:14}{n:>14}{100 * (n + other_ok) / n_closed:>13.2f}%')
print(f'(기준 closed = {100 * (base_ok + other_ok) / n_closed:.2f}%, 선택형 {len(qf)}문항 정답 {other_ok}개 포함)')
print('※ test 기반 탐색입니다. prior 규칙은 test 정답을 쓰지 않지만 선택지 순서·프롬프트를 test 로 골랐으므로 규칙 준수 행과 별도로 표기하세요. oracle 은 상한 참고용이며 보고하지 마세요.')


yes/no 형태 closed 문항 248개 x 2가지 순서 = 496회 추론
generate 블록 2개 발견 -> 첫 번째만 교체
/content/LLaVA-Med
[  0.0분] 진행 0/496
[  1.0분] 진행 85/496
[  2.0분] 진행 256/496
[  3.0분] 진행 426/496
추론 완료 (4.0분)
첫 답 위치 확인 (t: 답 토큰 위치, head: 모델 출력 앞부분):
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
   3 ' Assistant: no\n##'
train 의 yes 비율 0.498 (yes/no 정답 942개) | 테스트 yes/no 문항 중 정답 yes 비율 0.472

yes/no 형태 248문항 중 기준(접미어 [yes, no], greedy) 정답: 190개 (정답이 yes/no 인 문항 248개)
입력    규칙                yes/no 정답수   closed(272)
순서A   0                        190        73.90%
순서A   prior                    196        76.10%
순서A   oracle(상한)               199        77.21%
순서B   0                        186        72.43%
순서B   prior                    194        75.37%
순서B   oracle(상한)               196        76.10%
평균    0                        188        73.16%
평균    prior                    196        76.10%
평균    oracle(상한)               198        76.84%
(기준 closed = 73.90%, 선택형 24문항 정답 